# Relevant Imports

# Calculate the Realized Return

In [154]:
from pathlib import Path
import numpy as np, pandas as pd

ROOT   = Path("./")  # /ood_validation/macro_retrieval
TRAIND = ROOT / "train"

inp = TRAIND / "sp500_features_with_zretr.parquet"
out = TRAIND / "sp500_features_with_ret.parquet"

df = pd.read_parquet(inp).sort_values("Date").reset_index(drop=True)

# Preferred: compute same-day log return from Close/Open
if "Close" in df.columns:
    df["Daily_Return"] = np.log(df["Close"] / df["Open"]).replace([np.inf, -np.inf], np.nan)
else:
    # Fallback: shift lagged returns forward
    if "Daily_Return_lag1" not in df.columns:
        raise KeyError("Neither Close nor Daily_Return_lag1 found in train data.")
    df["Daily_Return"] = df["Daily_Return_lag1"].shift(-1)

# Drop trailing NaNs (from shift edge)
nan_tail = int(df["Daily_Return"].isna().sum())
if nan_tail:
    df = df.iloc[:-nan_tail].copy()

df.to_parquet(out, index=False)
print("TRAIN with realized return saved:", out, "| rows:", len(df))
print(df[["Date","Daily_Return"]].head())


TRAIN with realized return saved: train/sp500_features_with_ret.parquet | rows: 2798
        Date  Daily_Return
0 2007-08-07      0.476339
1 2007-08-10     -1.208174
2 2007-08-13     -0.102920
3 2007-08-14     -1.393884
4 2007-08-15     -1.083829


In [160]:
from pathlib import Path
import numpy as np, pandas as pd

# --- Direct absolute paths ---
inp  = Path("x_test_ood.parquet")
base = Path("x_test_ood_base.parquet")

# Use whichever exists
if inp.exists():
    infile = inp
else:
    infile = base

outfile = infile.with_name(infile.stem + "_with_ret.parquet")

# --- Load ---
df = pd.read_parquet(infile).sort_values("Date").reset_index(drop=True)

# --- Compute realized returns ---
if "Close" in df.columns and "Open" in df.columns:
    df["Daily_Return"] = np.log(df["Close"] / df["Open"]).replace([np.inf, -np.inf], np.nan)
elif "Daily_Return_lag1" in df.columns:
    df["Daily_Return"] = df["Daily_Return_lag1"].shift(-1)
else:
    raise KeyError("Neither (Close, Open) nor Daily_Return_lag1 found in OOD data.")

# --- Drop trailing NaNs ---
nan_tail = int(df["Daily_Return"].isna().sum())
if nan_tail:
    df = df.iloc[:-nan_tail].copy()

# --- Save ---
df.to_parquet(outfile, index=False)
print("OOD with realized return saved:", outfile, "| rows:", len(df))
print(df[["Date","Daily_Return"]].head())


OOD with realized return saved: x_test_ood_with_ret.parquet | rows: 228
        Date  Daily_Return
0 2024-01-09     -0.196457
1 2024-01-10      0.844844
2 2024-01-11      0.313409
3 2024-01-12     -2.372397
4 2024-01-16     -1.184751


In [182]:
import pandas as pd
from pathlib import Path

# --- Direct absolute paths to your outputs ---
train_file = Path("train/sp500_features_with_ret.parquet")
ood_file   = Path("x_test_ood_with_ret.parquet")  # or adjust name if base was used

for f in [train_file, ood_file]:
    if f.exists():
        df = pd.read_parquet(f)
        print("\n=== File:", f, "===")
        print("Columns:", list(df.columns))
        print("'z_retr' present? ->", "z_retr" in df.columns)
        print("Rows:", len(df))
        # Peek first few rows for sanity
        print(df.head(2))
    else:
        print("\n[WARN] File not found:", f)



=== File: train/sp500_features_with_ret.parquet ===
Columns: ['Date', 'Movement', 'Open', 'Close_lag1', 'High_lag1', 'Volume_lag1', 'Daily_Return_lag1', 'Volatility_lag1', 'sentiment_volatility_lag1', 'aggregate_sentiment_score_lag1', 'text_embed', 'cpi_yoy_lagged_z', 'unrate_lagged_z', 't10y2y_lagged_z', 'gdp_qoq_lagged_z', 'z_retr', 'Daily_Return']
'z_retr' present? -> True
Rows: 2798
        Date  Movement      Open  Close_lag1  High_lag1  Volume_lag1  Daily_Return_lag1  Volatility_lag1  sentiment_volatility_lag1  \
0 2007-08-07         1 -1.876038   -1.877060  -1.882864     1.016253           0.462693         0.214576                  -1.345383   
1 2007-08-10         0 -1.891313   -1.865612  -1.862054     0.671365           0.476339         0.356410                  -1.365977   

   aggregate_sentiment_score_lag1                                         text_embed  cpi_yoy_lagged_z  unrate_lagged_z  t10y2y_lagged_z  gdp_qoq_lagged_z  \
0                       -0.149489  [-0.077201

# Experiments

# Numeircal Only

In [162]:
from pathlib import Path
import pandas as pd

ROOT   = Path("./")  # notebook at /ood_validation/macro_retrieval
TRAIND = ROOT / "train"
TESTD  = ROOT / "test"

# Prefer files with realized-return; fallback to originals
def pick_train_path():
    p1 = TRAIND / "sp500_features_with_ret.parquet"
    p0 = TRAIND / "sp500_features.parquet"
    return p1 if p1.exists() else p0

def pick_ood_path():
    # prefer OOD with retrieval; else base; both prefer *_with_ret if available
    cands = [
        TESTD / "x_test_ood_with_ret.parquet",
        TESTD / "x_test_ood.parquet",
        TESTD / "x_test_ood_base_with_ret.parquet",
        TESTD / "x_test_ood_base.parquet",
    ]
    for p in cands:
        if p.exists(): return p
    raise FileNotFoundError("No OOD parquet found in /test.")

TRAIN_PARQUET = pick_train_path()
OOD_PARQUET   = pick_ood_path()

print("Using TRAIN:", TRAIN_PARQUET)
print("Using OOD  :", OOD_PARQUET)

train = pd.read_parquet(TRAIN_PARQUET).sort_values("Date").reset_index(drop=True)
ood   = pd.read_parquet(OOD_PARQUET).sort_values("Date").reset_index(drop=True)


Using TRAIN: train/sp500_features_with_ret.parquet
Using OOD  : test/x_test_ood_with_ret.parquet


In [163]:
import numpy as np

def trading_metrics(y_true, y_pred, realized_ret):
    if realized_ret is None or len(realized_ret) == 0:
        return {"win_rate": None, "profit_factor": None, "sharpe_252": None}
    pos = np.where(np.asarray(y_pred) > 0, 1.0, -1.0)
    strat_ret = pos * np.asarray(realized_ret)
    win_rate = (strat_ret > 0).mean()
    gross_profit = strat_ret[strat_ret > 0].sum()
    gross_loss   = -strat_ret[strat_ret < 0].sum()
    profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.inf
    mu, sd = strat_ret.mean(), strat_ret.std(ddof=1)
    sharpe_252 = (mu / sd) * np.sqrt(252) if sd > 0 else np.nan
    return {"win_rate": float(win_rate), "profit_factor": float(profit_factor), "sharpe_252": float(sharpe_252)}

def realized_return_column(df):
    # Prefer explicitly-added realized return
    if "Daily_Return" in df.columns and pd.api.types.is_numeric_dtype(df["Daily_Return"]):
        return "Daily_Return"
    # Fallbacks (rare)
    for c in ["ret", "RET", "strategy_return", "realized_return"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None


In [164]:
def pick_numeric_only_columns(df):
    drop_exact = {
        "Date","Movement","text_embed","z_retr",
        "cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z",
        "Daily_Return"  # never use realized return as a feature
    }
    keep = [c for c in df.columns if c not in drop_exact and pd.api.types.is_numeric_dtype(df[c])]
    return keep

X_cols = pick_numeric_only_columns(train)
print("Numeric-only features:", len(X_cols))


Numeric-only features: 8


In [165]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score
)
import numpy as np
import pandas as pd

# Features/labels
X_cols = pick_numeric_only_columns(train)
y_col  = "Movement"

print("Numeric-only features:", len(X_cols))
print("Train rows:", len(train), "Train date range:",
      train["Date"].min().date(), "→", train["Date"].max().date())

# CV setup
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df = train.iloc[tr_idx].copy()
    va_df = train.iloc[va_idx].copy()

    X_tr, y_tr = tr_df[X_cols].to_numpy(), tr_df[y_col].to_numpy()
    X_va, y_va = va_df[X_cols].to_numpy(), va_df[y_col].to_numpy()

    # Pipeline: scale numerics → LR
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(max_iter=500, solver="liblinear", n_jobs=1, random_state=42))
    ])
    pipe.fit(X_tr, y_tr)

    proba_va = pipe.predict_proba(X_va)[:, 1]
    yhat_va  = (proba_va >= 0.5).astype(int)

    # Classification metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try:
        auroc = roc_auc_score(y_va, proba_va)
    except ValueError:
        auroc = np.nan

    # Trading metrics (if realized return exists)
    rr_col = realized_return_column(va_df)
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "n_val": len(va_df),
        "val_start": va_df["Date"].min().date(),
        "val_end": va_df["Date"].max().date(),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"],
        "ret_col": rr_col
    })

cv_df = pd.DataFrame(cv_rows)
print("\n=== CV results (per fold) ===")
display(cv_df)

print("\n=== CV means ===")
display(cv_df[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]]
        .mean(numeric_only=True))


Numeric-only features: 8
Train rows: 2798 Train date range: 2007-08-07 → 2023-07-13

=== CV results (per fold) ===


,fold,n_val,val_start,val_end,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252,ret_col
0,1,466,2012-09-11,2015-12-04,0.699571,0.910959,0.511538,0.655172,0.480137,0.842942,0.630901,2.604534,5.376985,Daily_Return
1,2,466,2015-12-07,2017-10-18,0.581545,0.573298,0.872510,0.691943,0.148320,0.652960,0.508584,1.069039,0.360164,Daily_Return
2,3,466,2017-10-19,2019-10-04,0.637339,0.617647,1.000000,0.763636,0.277139,0.878912,0.508584,1.084499,0.443799,Daily_Return
3,4,466,2019-10-11,2021-08-27,0.684549,0.659722,1.000000,0.794979,0.352031,0.865348,0.600858,2.537207,4.169414,Daily_Return
4,5,466,2021-08-30,2023-07-13,0.510730,0.502183,1.000000,0.668605,0.130473,0.882222,0.461373,0.938449,-0.384619,Daily_Return



=== CV means ===


Accuracy        0.622747
Precision       0.652762
Recall          0.876810
F1              0.714867
MCC             0.277620
AUROC           0.824477
WinRate         0.542060
ProfitFactor    1.646745
Sharpe_252      1.993149
dtype: float64

In [166]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score
)
import numpy as np

# Train full model on TRAIN
pipe_full = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", LogisticRegression(max_iter=1000, solver="liblinear", n_jobs=1, random_state=42))
])
pipe_full.fit(train[X_cols].to_numpy(), train["Movement"].to_numpy())

# Predict on OOD
print("OOD rows:", len(ood), "OOD date range:", ood["Date"].min().date(), "→", ood["Date"].max().date())
X_test = ood[X_cols].to_numpy()
y_test = ood["Movement"].to_numpy()

proba_test = pipe_full.predict_proba(X_test)[:, 1]
yhat_test  = (proba_test >= 0.5).astype(int)

# Classification metrics
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try:
    auroc = roc_auc_score(y_test, proba_test)
except ValueError:
    auroc = np.nan

# Trading metrics on OOD
rr_col_test = realized_return_column(ood)
tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

print("\n=== OOD (AAPL 2024) — Numerical-only LR ===")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col_test:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col_test}")
else:
    print("No realized-return column found in OOD; trading metrics skipped.")


OOD rows: 228 OOD date range: 2024-01-09 → 2024-12-10

=== OOD (AAPL 2024) — Numerical-only LR ===
Accuracy: 0.4781 | Precision: 0.4533 | Recall: 0.9798 | F1: 0.6198 | MCC: 0.1503 | AUROC: 0.5672
WinRate: 0.4079 | ProfitFactor: 0.7575 | Sharpe_252: -1.5835 | ReturnCol: Daily_Return


# Text Only

In [167]:
def pick_text_only_columns(df):
    # text_embed is stored as list/array → expand into numpy
    if "text_embed" not in df.columns:
        raise KeyError("text_embed column not found in dataframe")
    return "text_embed"

X_col_text = pick_text_only_columns(train)
print("Using text-only features from column:", X_col_text)


Using text-only features from column: text_embed


In [168]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score
)

tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx], train.iloc[va_idx]

    # Expand embeddings to numpy
    X_tr = np.vstack(tr_df[X_col_text].to_numpy())
    X_va = np.vstack(va_df[X_col_text].to_numpy())
    y_tr, y_va = tr_df["Movement"].to_numpy(), va_df["Movement"].to_numpy()

    # Logistic Regression (no scaling needed for embeddings)
    clf = LogisticRegression(max_iter=500, solver="liblinear", n_jobs=1, random_state=42)
    clf.fit(X_tr, y_tr)

    proba_va = clf.predict_proba(X_va)[:, 1]
    yhat_va  = (proba_va >= 0.5).astype(int)

    # Classification metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try:
        auroc = roc_auc_score(y_va, proba_va)
    except ValueError:
        auroc = np.nan

    # Trading metrics
    rr_col = realized_return_column(va_df)
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "n_val": len(va_df),
        "val_start": va_df["Date"].min().date(), "val_end": va_df["Date"].max().date(),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"],
        "ret_col": rr_col
    })

cv_df = pd.DataFrame(cv_rows)
print("\n=== Text-only CV results (per fold) ===")
display(cv_df)

print("\n=== Text-only CV means ===")
display(cv_df[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]]
        .mean(numeric_only=True))



=== Text-only CV results (per fold) ===


,fold,n_val,val_start,val_end,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252,ret_col
0,1,466,2012-09-11,2015-12-04,0.510730,0.586022,0.419231,0.488789,0.046086,0.515161,0.506438,1.137495,0.736640,Daily_Return
1,2,466,2015-12-07,2017-10-18,0.540773,0.539785,1.000000,0.701117,0.050106,0.510238,0.442060,0.842384,-0.924745,Daily_Return
2,3,466,2017-10-19,2019-10-04,0.583691,0.588764,0.959707,0.729805,0.027355,0.473419,0.480687,0.873145,-0.741958,Daily_Return
3,4,466,2019-10-11,2021-08-27,0.603004,0.608225,0.985965,0.752343,-0.074153,0.523078,0.540773,1.022719,0.104220,Daily_Return
4,5,466,2021-08-30,2023-07-13,0.493562,0.493562,1.000000,0.660920,0.000000,0.492189,0.452790,0.842321,-1.038623,Daily_Return



=== Text-only CV means ===


Accuracy        0.546352
Precision       0.563272
Recall          0.872981
F1              0.666595
MCC             0.009879
AUROC           0.502817
WinRate         0.484549
ProfitFactor    0.943613
Sharpe_252     -0.372893
dtype: float64

In [169]:
# Full-train model
X_tr_full = np.vstack(train[X_col_text].to_numpy())
y_tr_full = train["Movement"].to_numpy()

clf_full = LogisticRegression(max_iter=1000, solver="liblinear", n_jobs=1, random_state=42)
clf_full.fit(X_tr_full, y_tr_full)

# OOD evaluation
X_test = np.vstack(ood[X_col_text].to_numpy())
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:, 1]
yhat_test  = (proba_test >= 0.5).astype(int)

# Metrics
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try:
    auroc = roc_auc_score(y_test, proba_test)
except ValueError:
    auroc = np.nan

rr_col_test = realized_return_column(ood)
tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

print("\n=== OOD (AAPL 2024) — Text-only LR ===")
print("Rows:", len(ood), "Date range:", ood["Date"].min().date(), "→", ood["Date"].max().date())
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col_test:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col_test}")
else:
    print("No realized-return column found in OOD; trading metrics skipped.")



=== OOD (AAPL 2024) — Text-only LR ===
Rows: 228 Date range: 2024-01-09 → 2024-12-10
Accuracy: 0.4342 | Precision: 0.4342 | Recall: 1.0000 | F1: 0.6055 | MCC: 0.0000 | AUROC: 0.4252
WinRate: 0.3991 | ProfitFactor: 0.6679 | Sharpe_252: -2.2977 | ReturnCol: Daily_Return


# 3. Multimodal

In [170]:
# Numeric (incl. macro_z) + text_embed, exclude labels/leakage columns
def pick_numeric_incl_macro(df):
    drop = {
        "Date","Movement","z_retr","Daily_Return",  # label/leakage
        # keep macro_z this time (unlike the numeric-only baseline)
    }
    keep = []
    for c in df.columns:
        if c in drop: 
            continue
        if c == "text_embed":
            continue  # handled separately
        if pd.api.types.is_numeric_dtype(df[c]):
            keep.append(c)
    return keep

num_cols_mm = pick_numeric_incl_macro(train)
text_col    = "text_embed"

print("Multimodal numeric+macro feature count:", len(num_cols_mm))
print("First 10 numeric/macro cols:", num_cols_mm[:10])
print("Using text column:", text_col)


Multimodal numeric+macro feature count: 12
First 10 numeric/macro cols: ['Open', 'Close_lag1', 'High_lag1', 'Volume_lag1', 'Daily_Return_lag1', 'Volatility_lag1', 'sentiment_volatility_lag1', 'aggregate_sentiment_score_lag1', 'cpi_yoy_lagged_z', 'unrate_lagged_z']
Using text column: text_embed


In [171]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score
)
import numpy as np
import pandas as pd

tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx], train.iloc[va_idx]

    # Split by modality
    Xnum_tr = tr_df[num_cols_mm].to_numpy(dtype=float)
    Xnum_va = va_df[num_cols_mm].to_numpy(dtype=float)
    Xtxt_tr = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
    Xtxt_va = np.vstack(va_df[text_col].to_numpy()).astype("float32")

    # Scale numerics (+macro_z) only; leave embeddings as-is
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xnum_tr_s = scaler.fit_transform(Xnum_tr)
    Xnum_va_s = scaler.transform(Xnum_va)

    # Concatenate: [scaled numerics+macro | text embedding]
    X_tr = np.hstack([Xnum_tr_s, Xtxt_tr]).astype("float32")
    X_va = np.hstack([Xnum_va_s, Xtxt_va]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy()
    y_va = va_df["Movement"].to_numpy()

    # LR classifier
    clf = LogisticRegression(max_iter=1000, solver="liblinear", n_jobs=1, random_state=42)
    clf.fit(X_tr, y_tr)

    proba_va = clf.predict_proba(X_va)[:, 1]
    yhat_va  = (proba_va >= 0.5).astype(int)

    # Classification metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try:
        auroc = roc_auc_score(y_va, proba_va)
    except ValueError:
        auroc = np.nan

    # Trading metrics
    rr_col = realized_return_column(va_df)
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "n_val": len(va_df),
        "val_start": va_df["Date"].min().date(), "val_end": va_df["Date"].max().date(),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"],
        "ret_col": rr_col
    })

cv_df_mm = pd.DataFrame(cv_rows)
print("\n=== Multimodal (No-Ret) CV results (per fold) ===")
display(cv_df_mm)

print("\n=== Multimodal (No-Ret) CV means ===")
display(cv_df_mm[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]]
        .mean(numeric_only=True))



=== Multimodal (No-Ret) CV results (per fold) ===


,fold,n_val,val_start,val_end,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252,ret_col
0,1,466,2012-09-11,2015-12-04,0.643777,0.614634,0.969231,0.752239,0.308889,0.750896,0.579399,1.781465,3.282091,Daily_Return
1,2,466,2015-12-07,2017-10-18,0.553648,0.548533,0.968127,0.700288,0.087211,0.616900,0.454936,0.906997,-0.526563,Daily_Return
2,3,466,2017-10-19,2019-10-04,0.667382,0.646040,0.956044,0.771049,0.311987,0.787318,0.525751,1.306655,1.461069,Daily_Return
3,4,466,2019-10-11,2021-08-27,0.667382,0.663317,0.926316,0.773060,0.256769,0.677678,0.566524,1.957009,3.058309,Daily_Return
4,5,466,2021-08-30,2023-07-13,0.620172,0.566085,0.986957,0.719493,0.360292,0.796002,0.532189,1.455227,2.268049,Daily_Return



=== Multimodal (No-Ret) CV means ===


Accuracy        0.630472
Precision       0.607722
Recall          0.961335
F1              0.743226
MCC             0.265030
AUROC           0.725759
WinRate         0.531760
ProfitFactor    1.481471
Sharpe_252      1.908591
dtype: float64

In [172]:
# Build full-train matrices
Xnum_full = train[num_cols_mm].to_numpy(dtype=float)
Xtxt_full = np.vstack(train[text_col].to_numpy()).astype("float32")
scaler_full = StandardScaler(with_mean=True, with_std=True)
Xnum_full_s = scaler_full.fit_transform(Xnum_full)
X_full = np.hstack([Xnum_full_s, Xtxt_full]).astype("float32")
y_full = train["Movement"].to_numpy()

clf_full = LogisticRegression(max_iter=2000, solver="liblinear", n_jobs=1, random_state=42)
clf_full.fit(X_full, y_full)

# OOD matrices
Xnum_test = ood[num_cols_mm].to_numpy(dtype=float)
Xtxt_test = np.vstack(ood[text_col].to_numpy()).astype("float32")
Xnum_test_s = scaler_full.transform(Xnum_test)
X_test = np.hstack([Xnum_test_s, Xtxt_test]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:, 1]
yhat_test  = (proba_test >= 0.5).astype(int)

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score
)
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try:
    auroc = roc_auc_score(y_test, proba_test)
except ValueError:
    auroc = np.nan

rr_col_test = realized_return_column(ood)
tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

print("\n=== OOD (AAPL 2024) — Multimodal (No-Ret) LR ===")
print("Rows:", len(ood), "Date range:", ood["Date"].min().date(), "→", ood["Date"].max().date())
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col_test:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col_test}")
else:
    print("No realized-return column found in OOD; trading metrics skipped.")



=== OOD (AAPL 2024) — Multimodal (No-Ret) LR ===
Rows: 228 Date range: 2024-01-09 → 2024-12-10
Accuracy: 0.4737 | Precision: 0.4371 | Recall: 0.7374 | F1: 0.5489 | MCC: 0.0097 | AUROC: 0.4960
WinRate: 0.4386 | ProfitFactor: 0.9974 | Sharpe_252: -0.0147 | ReturnCol: Daily_Return


# Retrieval Code

# Text Retrieval

In [173]:
import os, time
os.environ.setdefault("OMP_NUM_THREADS","1"); os.environ.setdefault("MKL_NUM_THREADS","1")

import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

ROOT   = Path("./")            # notebook at /ood_validation/macro_retrieval
TRAIND = ROOT / "train"
TESTD  = ROOT / "test"

# prefer files with realized returns
TRAIN_PARQUET = (TRAIND/"sp500_features_with_ret.parquet") if (TRAIND/"sp500_features_with_ret.parquet").exists() else (TRAIND/"sp500_features.parquet")
OOD_PARQUET_CAND = [
    TESTD/"x_test_ood_alpha0_with_ret.parquet",
    TESTD/"x_test_ood_with_ret.parquet",
    TESTD/"x_test_ood_alpha0.parquet",
    TESTD/"x_test_ood.parquet",
    TESTD/"x_test_ood_base_with_ret.parquet",
    TESTD/"x_test_ood_base.parquet",
]
for _p in OOD_PARQUET_CAND:
    if _p.exists():
        OOD_PARQUET = _p; break
else:
    raise FileNotFoundError("No OOD parquet found in /test.")

print("TRAIN:", TRAIN_PARQUET)
print("OOD  :", OOD_PARQUET)

train = pd.read_parquet(TRAIN_PARQUET).sort_values("Date").reset_index(drop=True)
ood   = pd.read_parquet(OOD_PARQUET).sort_values("Date").reset_index(drop=True)

def realized_return_column(df):
    if "Daily_Return" in df.columns and pd.api.types.is_numeric_dtype(df["Daily_Return"]): return "Daily_Return"
    for c in ["ret","RET","strategy_return","realized_return"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]): return c
    return None

def trading_metrics(y_true, y_pred, realized_ret):
    if realized_ret is None or len(realized_ret)==0:
        return {"win_rate": None, "profit_factor": None, "sharpe_252": None}
    pos = np.where(np.asarray(y_pred)>0, 1.0, -1.0)
    strat_ret = pos * np.asarray(realized_ret)
    win_rate = float((strat_ret>0).mean())
    gp = strat_ret[strat_ret>0].sum(); gl = -strat_ret[strat_ret<0].sum()
    pf = float(gp/gl) if gl>0 else np.inf
    mu, sd = strat_ret.mean(), strat_ret.std(ddof=1)
    sharpe = float((mu/sd)*np.sqrt(252)) if sd>0 else np.nan
    return {"win_rate":win_rate,"profit_factor":pf,"sharpe_252":sharpe}

# --- feature selectors ---
def pick_text_col(df):
    if "text_embed" not in df.columns: raise KeyError("text_embed missing")
    return "text_embed"

def pick_numeric_no_macro(df):
    drop = {"Date","Movement","text_embed","z_retr",
            "cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z",
            "Daily_Return"}
    keep = [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]
    return keep

text_col = pick_text_col(train)
num_cols = pick_numeric_no_macro(train)
print("Numerics (no macro_z) count:", len(num_cols), "| text col:", text_col)


TRAIN: train/sp500_features_with_ret.parquet
OOD  : test/x_test_ood_with_ret.parquet
Numerics (no macro_z) count: 8 | text col: text_embed


In [174]:
import faiss

def to_unit_rows(M):
    M = M.astype("float32")
    nrm = np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M / nrm

def build_faiss_index(vecs, use_gpu=True):
    # IndexFlatIP over L2-normalized vectors => cosine similarity
    d = vecs.shape[1]
    cpu_index = faiss.IndexFlatIP(d)
    if use_gpu and faiss.get_num_gpus()>0:
        res = faiss.StandardGpuResources()
        return faiss.index_cpu_to_gpu(res, 0, cpu_index), True
    return cpu_index, False

def compute_zretr(query_text, ref_text, ref_dates, query_dates, K=5, batch=512, use_gpu=True):
    """
    α=0 -> queries = normalized text only. ref_text are normalized text of ref set.
    Causal mask: ref_date < query_date. Aggregate neighbor TEXT into z_retr (mean).
    """
    q = to_unit_rows(query_text)
    r = to_unit_rows(ref_text)

    index, gpu = build_faiss_index(r, use_gpu=use_gpu)
    index.add(r)

    z = np.zeros((q.shape[0], r.shape[1]), dtype="float32")
    effk = []
    for s in range(0, q.shape[0], batch):
        e = min(s+batch, q.shape[0])
        D, I = index.search(q[s:e], K*10)  # oversample for mask
        for i in range(s, e):
            cand = I[i-s]
            mask = (ref_dates[cand] < query_dates[i])
            pick = np.where(mask)[0][:K]
            if pick.size==0:  # fallback
                pick = np.arange(min(K, len(cand)))
            ids = cand[pick]
            z[i] = r[ids].mean(axis=0)  # mean of TEXT neighbors
            effk.append(len(ids))
    return z, np.array(effk, dtype=int)


In [176]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    # --- Build reference (fold-train) once for this fold ---
    ref_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
    ref_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

    # --- z_retr for TRAIN (causal within fold-train) ---
    z_tr, effk_tr = compute_zretr(
        query_text = ref_text,
        ref_text   = ref_text,
        ref_dates  = ref_dates,
        query_dates= ref_dates,
        K=5, batch=512, use_gpu=True
    )

    # --- z_retr for VAL (query fold-val against fold-train reference) ---
    qry_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
    qry_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")
    z_va, effk_va = compute_zretr(
        query_text = qry_text,
        ref_text   = ref_text,
        ref_dates  = ref_dates,
        query_dates= qry_dates,
        K=5, batch=512, use_gpu=True
    )

    # --- Numerics (no macro) scaling ---
    Xnum_tr = tr_df[num_cols].to_numpy(dtype=float)
    Xnum_va = va_df[num_cols].to_numpy(dtype=float)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xnum_tr_s = scaler.fit_transform(Xnum_tr)
    Xnum_va_s = scaler.transform(Xnum_va)

    # --- Text ---
    Xtxt_tr = ref_text
    Xtxt_va = qry_text

    # --- Final design matrices: [scaled numerics | text | z_retr] ---
    X_tr = np.hstack([Xnum_tr_s, Xtxt_tr, z_tr]).astype("float32")
    X_va = np.hstack([Xnum_va_s, Xtxt_va, z_va]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy()
    y_va = va_df["Movement"].to_numpy()

    # --- Classifier ---
    clf = LogisticRegression(max_iter=1000, solver="liblinear", n_jobs=1, random_state=42)
    clf.fit(X_tr, y_tr)

    proba_va = clf.predict_proba(X_va)[:, 1]
    yhat_va  = (proba_va >= 0.5).astype(int)

    # --- Metrics ---
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try:
        auroc = roc_auc_score(y_va, proba_va)
    except ValueError:
        auroc = np.nan
    rr_col = realized_return_column(va_df)
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "n_val": len(va_df),
        "val_start": va_df["Date"].min().date(), "val_end": va_df["Date"].max().date(),
        "effK_tr_min": int(effk_tr.min()), "effK_tr_med": float(np.median(effk_tr)), "effK_tr_max": int(effk_tr.max()),
        "effK_va_min": int(effk_va.min()), "effK_va_med": float(np.median(effk_va)), "effK_va_max": int(effk_va.max()),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"]
    })

cv_textret = pd.DataFrame(cv_rows)
print("\n=== Text-Ret (α=0) CV results (per fold) ===")
display(cv_textret)

print("\n=== Text-Ret (α=0) CV means ===")
display(cv_textret[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]]
        .mean(numeric_only=True))



=== Text-Ret (α=0) CV results (per fold) ===


,fold,n_val,val_start,val_end,effK_tr_min,effK_tr_med,effK_tr_max,effK_va_min,effK_va_med,effK_va_max,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252
0,1,466,2012-09-11,2015-12-04,1,5.0,5,5,5.0,5,0.639485,0.794872,0.476923,0.596154,0.338431,0.758122,0.622318,2.264893,4.615849
1,2,466,2015-12-07,2017-10-18,1,5.0,5,5,5.0,5,0.542918,0.542222,0.972112,0.696148,0.038252,0.624164,0.448498,0.817973,-1.082990
2,3,466,2017-10-19,2019-10-04,1,5.0,5,5,5.0,5,0.637339,0.622642,0.967033,0.757532,0.237403,0.780599,0.517167,1.242663,1.187464
3,4,466,2019-10-11,2021-08-27,1,5.0,5,5,5.0,5,0.673820,0.652874,0.996491,0.788889,0.317312,0.827450,0.590129,2.266282,3.693483
4,5,466,2021-08-30,2023-07-13,1,5.0,5,5,5.0,5,0.515021,0.504386,1.000000,0.670554,0.146193,0.841802,0.457082,0.896195,-0.663490



=== Text-Ret (α=0) CV means ===


Accuracy        0.601717
Precision       0.623399
Recall          0.882512
F1              0.701855
MCC             0.215518
AUROC           0.766427
WinRate         0.527039
ProfitFactor    1.497601
Sharpe_252      1.550064
dtype: float64

In [177]:
# --- Full-train z_retr (causal within full train) ---
ref_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
ref_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

z_tr_full, effk_full = compute_zretr(
    query_text = ref_text_full,
    ref_text   = ref_text_full,
    ref_dates  = ref_dates_full,
    query_dates= ref_dates_full,
    K=5, batch=1024, use_gpu=True
)
print("Full-train effK stats:", effk_full.min(), np.median(effk_full), effk_full.max())

# --- Build full-train matrices ---
Xnum_full = train[num_cols].to_numpy(dtype=float)
scaler_full = StandardScaler(with_mean=True, with_std=True).fit(Xnum_full)
Xnum_full_s = scaler_full.transform(Xnum_full)
Xtxt_full   = ref_text_full

X_full = np.hstack([Xnum_full_s, Xtxt_full, z_tr_full]).astype("float32")
y_full = train["Movement"].to_numpy()

clf_full = LogisticRegression(max_iter=2000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

# --- OOD: compute z_retr α=0 against FULL TRAIN (causal to OOD dates) ---
qry_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
qry_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

z_ood, effk_ood = compute_zretr(
    query_text = qry_text_ood,
    ref_text   = ref_text_full,
    ref_dates  = ref_dates_full,
    query_dates= qry_dates_ood,
    K=5, batch=1024, use_gpu=True
)
print("OOD effK stats:", effk_ood.min(), np.median(effk_ood), effk_ood.max())

# --- OOD design matrix ---
Xnum_test = ood[num_cols].to_numpy(dtype=float)
Xtxt_test = qry_text_ood
X_test = np.hstack([scaler_full.transform(Xnum_test), Xtxt_test, z_ood]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:, 1]
yhat_test  = (proba_test >= 0.5).astype(int)

# --- Metrics ---
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try:
    auroc = roc_auc_score(y_test, proba_test)
except ValueError:
    auroc = np.nan

rr_col_test = realized_return_column(ood)
tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

print("\n=== OOD (AAPL 2024) — Text-Ret (α=0) LR ===")
print("Rows:", len(ood), "Date range:", ood["Date"].min().date(), "→", ood["Date"].max().date())
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col_test:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col_test}")
else:
    print("No realized-return column found in OOD; trading metrics skipped.")


Full-train effK stats: 1 5.0 5
OOD effK stats: 5 5.0 5

=== OOD (AAPL 2024) — Text-Ret (α=0) LR ===
Rows: 228 Date range: 2024-01-09 → 2024-12-10
Accuracy: 0.4298 | Precision: 0.4286 | Recall: 0.9394 | F1: 0.5886 | MCC: -0.0505 | AUROC: 0.5231
WinRate: 0.4035 | ProfitFactor: 0.7508 | Sharpe_252: -1.6341 | ReturnCol: Daily_Return


# Macro Retrieval

In [185]:
# Config
ALPHA = 0.5        # try {0.25, 0.5, 1.0}
K = 5              # try {3,5,10}
BATCH = 512
USE_GPU = True

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col  = "text_embed"

# --- Use precomputed z_retr from parquet (no FAISS) ---
USE_PRECOMPUTED_Z = True

def z_from(df):
    if "z_retr" not in df.columns:
        raise KeyError("Expected 'z_retr' column in dataframe but not found.")
    Z = np.vstack(df["z_retr"].to_numpy()).astype("float32")
    # tiny sanity
    if np.isnan(Z).any():
        raise ValueError("NaNs found in precomputed z_retr.")
    return Z


def to_unit_rows(M):
    M = M.astype("float32")
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

def make_joint(text_mat, macro_mat, alpha):
    # [text ; α * macro] then L2-normalize → dim = 384 + 4
    joint = np.concatenate([text_mat.astype("float32"), (alpha * macro_mat.astype("float32"))], axis=1)
    return to_unit_rows(joint)

import faiss
def build_flat_ip(d, use_gpu=True):
    idx = faiss.IndexFlatIP(d)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        return faiss.index_cpu_to_gpu(res, 0, idx), True
    return idx, False

def z_retr_from_joint(q_joint, r_joint, r_text, r_dates, q_dates, K=5, batch=512, use_gpu=True):
    # r_joint is the FAISS index vectors (normalized joint); r_text aggregated for z_retr
    d = r_joint.shape[1]
    index, gpu = build_flat_ip(d, use_gpu)
    index.add(r_joint)

    Z = np.zeros((q_joint.shape[0], r_text.shape[1]), dtype="float32")
    effK = []

    for s in range(0, q_joint.shape[0], batch):
        e = min(s+batch, q_joint.shape[0])
        D, I = index.search(q_joint[s:e], K*10)  # oversample then causal-mask
        for i in range(s, e):
            cand = I[i-s]
            mask = (r_dates[cand] < q_dates[i])
            pick = np.where(mask)[0][:K]
            if pick.size == 0:
                pick = np.arange(min(K, len(cand)))
            ids = cand[pick]
            Z[i] = r_text[ids].mean(axis=0)
            effK.append(len(ids))
    return Z, np.array(effK, dtype=int)


In [189]:
# Use numerics + macro_z
def pick_numeric_incl_macro(df):
    drop = {"Date","Movement","z_retr","Daily_Return","text_embed"}
    keep = [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]
    return keep

num_cols = pick_numeric_incl_macro(train)
print("Numerics+macro count:", len(num_cols))

tscv = TimeSeriesSplit(n_splits=5)
rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    # Reference (fold-train)
    R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
    R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
    R_joint = make_joint(R_text, R_macro, ALPHA)
    R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

    # TRAIN z_retr: use precomputed column
    if USE_PRECOMPUTED_Z:
        Ztr = z_from(tr_df)
        effK_tr = np.full(len(tr_df), K, dtype=int)  # placeholder stats (not applicable)
    else:
        Ztr, effK_tr = z_retr_from_joint(
            q_joint = R_joint, r_joint = R_joint, r_text = R_text,
            r_dates = R_dates, q_dates = R_dates,
            K=K, batch=BATCH, use_gpu=USE_GPU
        )
    print(f"[CV fold {fold}] Z source: {'precomputed parquet' if USE_PRECOMPUTED_Z else 'FAISS recompute'}")



    # VAL z_retr (query val against fold-train)
    Q_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
    Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
    Q_joint = make_joint(Q_text, Q_macro, ALPHA)
    Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

    # VAL z_retr: use precomputed column
    if USE_PRECOMPUTED_Z:
        Zva = z_from(va_df)
        effK_va = np.full(len(va_df), K, dtype=int)  # placeholder stats (not applicable)
    else:
        Zva, effK_va = z_retr_from_joint(
            q_joint = Q_joint, r_joint = R_joint, r_text = R_text,
            r_dates = R_dates, q_dates = Q_dates,
            K=K, batch=BATCH, use_gpu=USE_GPU
        )
    print(f"[CV fold {fold}] Z source: {'precomputed parquet' if USE_PRECOMPUTED_Z else 'FAISS recompute'}")



    # Build design matrices: [scaled numerics+macro | text | z_retr]
    Xnum_tr = tr_df[num_cols].to_numpy(dtype=float)
    Xnum_va = va_df[num_cols].to_numpy(dtype=float)
    scaler  = StandardScaler(with_mean=True, with_std=True)
    Xnum_tr_s = scaler.fit_transform(Xnum_tr)
    Xnum_va_s = scaler.transform(Xnum_va)

    Xtxt_tr = R_text
    Xtxt_va = Q_text

    X_tr = np.hstack([Xnum_tr_s, Xtxt_tr, Ztr]).astype("float32")
    X_va = np.hstack([Xnum_va_s, Xtxt_va, Zva]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy()
    y_va = va_df["Movement"].to_numpy()

    clf = LogisticRegression(max_iter=2000, solver="liblinear", n_jobs=1, random_state=42)
    clf.fit(X_tr, y_tr)

    proba_va = clf.predict_proba(X_va)[:, 1]
    yhat_va  = (proba_va >= 0.5).astype(int)

    # Metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try:
        auroc = roc_auc_score(y_va, proba_va)
    except ValueError:
        auroc = np.nan
    rr_col = realized_return_column(va_df)
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    rows.append({
        "fold": fold, "n_val": len(va_df),
        "effK_tr_min": int(effK_tr.min()), "effK_tr_med": float(np.median(effK_tr)), "effK_tr_max": int(effK_tr.max()),
        "effK_va_min": int(effK_va.min()), "effK_va_med": float(np.median(effK_va)), "effK_va_max": int(effK_va.max()),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"],
        "val_start": va_df["Date"].min().date(), "val_end": va_df["Date"].max().date()
    })

cv_macroret = pd.DataFrame(rows)
print("\n=== Macro-Ret (α={}) CV results (per fold) ===".format(ALPHA))
display(cv_macroret)

print("\n=== Macro-Ret (α={}) CV means ===".format(ALPHA))
display(cv_macroret[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]]
        .mean(numeric_only=True))


Numerics+macro count: 12
[CV fold 1] Z source: precomputed parquet
[CV fold 1] Z source: precomputed parquet
[CV fold 2] Z source: precomputed parquet
[CV fold 2] Z source: precomputed parquet
[CV fold 3] Z source: precomputed parquet
[CV fold 3] Z source: precomputed parquet
[CV fold 4] Z source: precomputed parquet
[CV fold 4] Z source: precomputed parquet
[CV fold 5] Z source: precomputed parquet
[CV fold 5] Z source: precomputed parquet

=== Macro-Ret (α=0.5) CV results (per fold) ===


,fold,n_val,effK_tr_min,effK_tr_med,effK_tr_max,effK_va_min,effK_va_med,effK_va_max,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252,val_start,val_end
0,1,466,5,5.0,5,5,5.0,5,0.624464,0.602410,0.961538,0.740741,0.255428,0.745967,0.572961,1.730950,3.120545,2012-09-11,2015-12-04
1,2,466,5,5.0,5,5,5.0,5,0.555794,0.549550,0.972112,0.702158,0.098435,0.602372,0.457082,0.910925,-0.503264,2015-12-07,2017-10-18
2,3,466,5,5.0,5,5,5.0,5,0.660944,0.641278,0.956044,0.767647,0.295615,0.767693,0.515021,1.212112,1.051666,2017-10-19,2019-10-04
3,4,466,5,5.0,5,5,5.0,5,0.665236,0.662469,0.922807,0.771261,0.250407,0.675565,0.564378,1.932783,3.003600,2019-10-11,2021-08-27
4,5,466,5,5.0,5,5,5.0,5,0.564378,0.531616,0.986957,0.691020,0.251851,0.802284,0.493562,1.195270,1.079621,2021-08-30,2023-07-13



=== Macro-Ret (α=0.5) CV means ===


Accuracy        0.614163
Precision       0.597464
Recall          0.959892
F1              0.734565
MCC             0.230347
AUROC           0.718776
WinRate         0.520601
ProfitFactor    1.396408
Sharpe_252      1.550434
dtype: float64

In [190]:
# Full-train reference
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_joint_full = make_joint(R_text_full, R_macro_full, ALPHA)
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

# Full-train z_retr: use precomputed column
if USE_PRECOMPUTED_Z:
    Ztr_full = z_from(train)
    effK_full = np.full(len(train), K, dtype=int)  # placeholder
else:
    Ztr_full, effK_full = z_retr_from_joint(
        q_joint = R_joint_full, r_joint = R_joint_full, r_text = R_text_full,
        r_dates = R_dates_full, q_dates = R_dates_full,
        K=K, batch=1024, use_gpu=USE_GPU
    )

print("Full-train effK stats:",
      effK_full.min() if len(effK_full) else 'NA',
      np.median(effK_full) if len(effK_full) else 'NA',
      effK_full.max() if len(effK_full) else 'NA')
print("[Full-train] Z source:", "precomputed parquet" if USE_PRECOMPUTED_Z else "FAISS recompute")


# Train matrix
Xnum_full  = train[num_cols].to_numpy(dtype=float)
scaler_full = StandardScaler(with_mean=True, with_std=True).fit(Xnum_full)
Xnum_full_s = scaler_full.transform(Xnum_full)
X_full = np.hstack([Xnum_full_s, R_text_full, Ztr_full]).astype("float32")
y_full = train["Movement"].to_numpy()

clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

# OOD z_retr with joint queries (macro-aware)
Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_joint_ood = make_joint(Q_text_ood, Q_macro_ood, ALPHA)
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

# OOD z_retr: use precomputed column
if USE_PRECOMPUTED_Z:
    Z_ood = z_from(ood)
    effK_ood = np.full(len(ood), K, dtype=int)  # placeholder
else:
    Z_ood, effK_ood = z_retr_from_joint(
        q_joint = Q_joint_ood, r_joint = R_joint_full, r_text = R_text_full,
        r_dates = R_dates_full, q_dates = Q_dates_ood,
        K=K, batch=1024, use_gpu=USE_GPU
    )

print("OOD effK stats:",
      effK_ood.min() if len(effK_ood) else 'NA',
      np.median(effK_ood) if len(effK_ood) else 'NA',
      effK_ood.max() if len(effK_ood) else 'NA')
print("[OOD] Z source:", "precomputed parquet" if USE_PRECOMPUTED_Z else "FAISS recompute")


# OOD matrix
Xnum_test  = ood[num_cols].to_numpy(dtype=float)
Xnum_test_s = scaler_full.transform(Xnum_test)
X_test = np.hstack([Xnum_test_s, Q_text_ood, Z_ood]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:, 1]
yhat_test  = (proba_test >= 0.5).astype(int)

acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try:
    auroc = roc_auc_score(y_test, proba_test)
except ValueError:
    auroc = np.nan

rr_col_test = realized_return_column(ood)
tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

print("\n=== OOD (AAPL 2024) — Macro-Ret (α={}) LR ===".format(ALPHA))
print("Rows:", len(ood), "Date range:", ood["Date"].min().date(), "→", ood["Date"].max().date())
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col_test:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col_test}")
else:
    print("No realized-return column found in OOD; trading metrics skipped.")


Full-train effK stats: 5 5.0 5
[Full-train] Z source: precomputed parquet
OOD effK stats: 5 5.0 5
[OOD] Z source: precomputed parquet

=== OOD (AAPL 2024) — Macro-Ret (α=0.5) LR ===
Rows: 228 Date range: 2024-01-09 → 2024-12-10
Accuracy: 0.4518 | Precision: 0.4110 | Recall: 0.6061 | F1: 0.4898 | MCC: -0.0626 | AUROC: 0.4968
WinRate: 0.4693 | ProfitFactor: 1.1812 | Sharpe_252: 0.9505 | ReturnCol: Daily_Return


# Alpha Sweep (α ∈ {0.25, 0.5, 1.0}) × (K ∈ {3,5,10})

In [193]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

ALPHAS = [0.25, 0.5, 1.0]
KS     = [3, 5, 10]
BATCH  = 512
USE_GPU = True

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col   = "text_embed"

def pick_numeric_incl_macro(df):
    drop = {"Date","Movement","z_retr","Daily_Return","text_embed"}
    return [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]

num_cols = pick_numeric_incl_macro(train)

# Pre-materialize arrays once
R_TEXT_FULL  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_MACRO_FULL = train[MACRO_COLS].to_numpy().astype("float32")
R_DATES_FULL = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_TEXT_OOD   = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_MACRO_OOD  = ood[MACRO_COLS].to_numpy().astype("float32")
Q_DATES_OOD  = ood["Date"].to_numpy(dtype="datetime64[ns]")

def run_macroret(alpha, K):
    # ---- CV ----
    tscv = TimeSeriesSplit(n_splits=5)
    cv_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
        tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

        # Fold reference (train)
        R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
        R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
        R_joint = make_joint(R_text, R_macro, alpha)
        R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

        # z_retr for train (causal within train)
        Ztr, _ = z_retr_from_joint(R_joint, R_joint, R_text, R_dates, R_dates, K=K, batch=BATCH, use_gpu=USE_GPU)

        # z_retr for val (query val vs train)
        Q_text = np.vstack(va_df[text_col].to_numpy()).astype("float32")
        Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
        Q_joint = make_joint(Q_text, Q_macro, alpha)
        Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")
        Zva, _ = z_retr_from_joint(Q_joint, R_joint, R_text, R_dates, Q_dates, K=K, batch=BATCH, use_gpu=USE_GPU)

        # Design matrices: [scaled numerics+macro | text | z_retr]
        Xnum_tr = tr_df[num_cols].to_numpy(dtype=float)
        Xnum_va = va_df[num_cols].to_numpy(dtype=float)
        scaler  = StandardScaler(with_mean=True, with_std=True).fit(Xnum_tr)
        X_tr = np.hstack([scaler.transform(Xnum_tr),
                          R_text,
                          Ztr]).astype("float32")
        X_va = np.hstack([scaler.transform(Xnum_va),
                          Q_text,
                          Zva]).astype("float32")

        y_tr = tr_df["Movement"].to_numpy()
        y_va = va_df["Movement"].to_numpy()

        clf = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_tr, y_tr)
        proba_va = clf.predict_proba(X_va)[:, 1]
        yhat_va  = (proba_va >= 0.5).astype(int)

        # Metrics
        acc  = accuracy_score(y_va, yhat_va)
        prec = precision_score(y_va, yhat_va, zero_division=0)
        rec  = recall_score(y_va, yhat_va, zero_division=0)
        f1   = f1_score(y_va, yhat_va, zero_division=0)
        mcc  = matthews_corrcoef(y_va, yhat_va)
        try: auroc = roc_auc_score(y_va, proba_va)
        except ValueError: auroc = np.nan
        rr_col = realized_return_column(va_df)
        tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

        cv_metrics.append([acc,prec,rec,f1,mcc,auroc,tm["win_rate"],tm["profit_factor"],tm["sharpe_252"]])

    cv_mean = np.nanmean(np.array(cv_metrics, dtype=float), axis=0)
    (cv_acc,cv_prec,cv_rec,cv_f1,cv_mcc,cv_auroc,cv_wr,cv_pf,cv_sharpe) = cv_mean

    # ---- Full-train → OOD ----
    R_joint_full = make_joint(R_TEXT_FULL, R_MACRO_FULL, alpha)
    Ztr_full, _  = z_retr_from_joint(R_joint_full, R_joint_full, R_TEXT_FULL, R_DATES_FULL, R_DATES_FULL,
                                     K=K, batch=1024, use_gpu=USE_GPU)

    Xnum_full   = train[num_cols].to_numpy(dtype=float)
    scaler_full = StandardScaler(with_mean=True, with_std=True).fit(Xnum_full)
    X_full = np.hstack([scaler_full.transform(Xnum_full),
                        R_TEXT_FULL,
                        Ztr_full]).astype("float32")
    y_full = train["Movement"].to_numpy()

    clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

    # OOD z_retr (query OOD vs full train)
    Q_joint_ood = make_joint(Q_TEXT_OOD, Q_MACRO_OOD, alpha)
    Z_ood, _    = z_retr_from_joint(Q_joint_ood, R_joint_full, R_TEXT_FULL, R_DATES_FULL, Q_DATES_OOD,
                                    K=K, batch=1024, use_gpu=USE_GPU)

    Xnum_test = ood[num_cols].to_numpy(dtype=float)
    X_test = np.hstack([scaler_full.transform(Xnum_test),
                        Q_TEXT_OOD,
                        Z_ood]).astype("float32")
    y_test = ood["Movement"].to_numpy()

    proba_test = clf_full.predict_proba(X_test)[:, 1]
    yhat_test  = (proba_test >= 0.5).astype(int)

    acc  = accuracy_score(y_test, yhat_test)
    prec = precision_score(y_test, yhat_test, zero_division=0)
    rec  = recall_score(y_test, yhat_test, zero_division=0)
    f1   = f1_score(y_test, yhat_test, zero_division=0)
    mcc  = matthews_corrcoef(y_test, yhat_test)
    try: auroc = roc_auc_score(y_test, proba_test)
    except ValueError: auroc = np.nan
    rr_col_test = realized_return_column(ood)
    tm = trading_metrics(y_test, yhat_test, ood[rr_col_test].to_numpy() if rr_col_test else None)

    return {
        "alpha": alpha, "K": K,
        # CV
        "CV_Accuracy": cv_acc, "CV_Precision": cv_prec, "CV_Recall": cv_rec, "CV_F1": cv_f1,
        "CV_MCC": cv_mcc, "CV_AUROC": cv_auroc, "CV_WinRate": cv_wr,
        "CV_ProfitFactor": cv_pf, "CV_Sharpe_252": cv_sharpe,
        # OOD
        "OOD_Accuracy": acc, "OOD_Precision": prec, "OOD_Recall": rec, "OOD_F1": f1,
        "OOD_MCC": mcc, "OOD_AUROC": auroc,
        "OOD_WinRate": tm["win_rate"], "OOD_ProfitFactor": tm["profit_factor"], "OOD_Sharpe_252": tm["sharpe_252"]
    }

# ---- Run sweep ----
results = []
for a in ALPHAS:
    for k in KS:
        print(f"\n>>> Running Macro-Ret sweep: alpha={a}, K={k}")
        res = run_macroret(alpha=a, K=k)
        results.append(res)

sweep_df = pd.DataFrame(results)
print("\n=== Sweep summary (sorted by OOD Sharpe) ===")
display(sweep_df.sort_values("OOD_Sharpe_252", ascending=False))

print("\nTop-3 by OOD Profit Factor:")
display(sweep_df.sort_values("OOD_ProfitFactor", ascending=False).head(3)[
    ["alpha","K","OOD_ProfitFactor","OOD_Sharpe_252","OOD_WinRate","OOD_AUROC"]
])



>>> Running Macro-Ret sweep: alpha=0.25, K=3

>>> Running Macro-Ret sweep: alpha=0.25, K=5

>>> Running Macro-Ret sweep: alpha=0.25, K=10

>>> Running Macro-Ret sweep: alpha=0.5, K=3

>>> Running Macro-Ret sweep: alpha=0.5, K=5

>>> Running Macro-Ret sweep: alpha=0.5, K=10

>>> Running Macro-Ret sweep: alpha=1.0, K=3

>>> Running Macro-Ret sweep: alpha=1.0, K=5

>>> Running Macro-Ret sweep: alpha=1.0, K=10

=== Sweep summary (sorted by OOD Sharpe) ===


,alpha,K,CV_Accuracy,CV_Precision,CV_Recall,CV_F1,CV_MCC,CV_AUROC,CV_WinRate,CV_ProfitFactor,CV_Sharpe_252,OOD_Accuracy,OOD_Precision,OOD_Recall,OOD_F1,OOD_MCC,OOD_AUROC,OOD_WinRate,OOD_ProfitFactor,OOD_Sharpe_252
4,0.50,5,0.618884,0.600194,0.960240,0.736984,0.239658,0.716414,0.525322,1.431852,1.703799,0.451754,0.410959,0.606061,0.489796,-0.062596,0.496750,0.469298,1.181239,0.950499
5,0.50,10,0.619742,0.600764,0.960599,0.737507,0.240167,0.717962,0.527039,1.455600,1.810345,0.451754,0.418750,0.676768,0.517375,-0.047847,0.491896,0.451754,1.092330,0.504144
2,0.25,10,0.624893,0.602992,0.963731,0.740749,0.249122,0.719626,0.532189,1.467999,1.867518,0.460526,0.425926,0.696970,0.528736,-0.026187,0.494480,0.442982,1.060692,0.336387
7,1.00,5,0.621459,0.601784,0.961000,0.738499,0.243487,0.715460,0.532189,1.477182,1.887402,0.456140,0.423313,0.696970,0.526718,-0.034817,0.493618,0.447368,1.056401,0.313250
8,1.00,10,0.617167,0.599825,0.959763,0.736063,0.239638,0.711573,0.525322,1.414306,1.683076,0.469298,0.435294,0.747475,0.550186,0.003743,0.493775,0.442982,1.034693,0.194717
1,0.25,5,0.621459,0.599563,0.968341,0.739638,0.248444,0.716861,0.530472,1.432036,1.726403,0.469298,0.434524,0.737374,0.546816,0.001058,0.498551,0.442982,1.033964,0.190691
0,0.25,3,0.625322,0.603213,0.959457,0.739861,0.248864,0.713879,0.532618,1.454241,1.786797,0.478070,0.439759,0.737374,0.550943,0.018317,0.488842,0.442982,1.024925,0.140563
6,1.00,3,0.624464,0.603086,0.962341,0.740235,0.249380,0.714215,0.534335,1.491258,1.912502,0.451754,0.414474,0.636364,0.501992,-0.056314,0.492757,0.451754,1.021846,0.123386
3,0.50,3,0.627039,0.604180,0.962419,0.741421,0.255820,0.717239,0.529185,1.441731,1.737778,0.486842,0.423729,0.505051,0.460829,-0.021903,0.494323,0.478070,0.995253,-0.027166



Top-3 by OOD Profit Factor:


,alpha,K,OOD_ProfitFactor,OOD_Sharpe_252,OOD_WinRate,OOD_AUROC
4,0.50,5,1.181239,0.950499,0.469298,0.496750
5,0.50,10,1.092330,0.504144,0.451754,0.491896
2,0.25,10,1.060692,0.336387,0.442982,0.494480


# Improvements

# Macro gate + Recency filter

In [115]:
# Gates (tweak later if needed)
TAU_MACRO = 0.30       # macro cosine threshold (try 0.2 / 0.3 / 0.4)
RECENCY_YEARS = 8      # only consider neighbors within last N years
K = 5                  # neighbors to aggregate (keep as in your best config)
ALPHA = 0.5            # same alpha for joint query
BATCH = 512
USE_GPU = True
MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col  = "text_embed"


In [116]:
import numpy as np, faiss
from datetime import timedelta

def to_unit_rows(M: np.ndarray) -> np.ndarray:
    M = M.astype("float32")
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

def make_joint(text_mat, macro_mat, alpha):
    joint = np.concatenate([text_mat.astype("float32"),
                            (alpha * macro_mat.astype("float32"))], axis=1)
    return to_unit_rows(joint)

def build_flat_ip(d, use_gpu=True):
    idx = faiss.IndexFlatIP(d)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        idx = faiss.index_cpu_to_gpu(res, 0, idx)
    return idx

def cosine_rows(A, B):
    # A [m,d], B [n,d] assumed L2-normalized
    return A @ B.T

def z_retr_with_gates(
    q_text, q_macro, q_dates,
    r_text, r_macro, r_dates,
    alpha=0.5, K=5, tau_macro=0.3, recency_years=8,
    batch=512, use_gpu=True, oversample=50
):
    """
    Step 1: dense search on joint vectors to get M=K*oversample candidates.
    Step 2: causal mask (r_date < q_date).
    Step 3: macro gate (cos(macro_q, macro_r) >= tau_macro).
    Step 4: recency gate (r_date >= q_date - N years).
    Aggregate remaining top-K by mean of r_text; fallback if <K.
    """
    # Prepare normalized blocks
    R_joint = make_joint(r_text, r_macro, alpha)
    Q_joint = make_joint(q_text, q_macro, alpha)
    R_text_n = to_unit_rows(r_text)
    R_macro_n = to_unit_rows(r_macro)
    Q_macro_n = to_unit_rows(q_macro)

    # FAISS on joint
    index = build_flat_ip(R_joint.shape[1], use_gpu)
    index.add(R_joint)

    Z = np.zeros((Q_joint.shape[0], r_text.shape[1]), dtype="float32")
    effK, kept_frac = [], []

    # Precompute macro cosine blocks if you like; here compute on the fly per batch
    for s in range(0, Q_joint.shape[0], batch):
        e = min(s+batch, Q_joint.shape[0])
        # get oversampled candidates
        D, I = index.search(Q_joint[s:e], K*oversample)

        # macro cosines for this batch vs all refs (vectorized)
        # To avoid huge matmul, we only compute macro cos with the candidate set
        for i in range(s, e):
            cand = I[i-s]                                # candidate ids
            qd   = q_dates[i]                            # query date
            # causal
            causal_mask = (r_dates[cand] < qd)
            # recency
            recency_cut = (qd - np.timedelta64(int(365.25*recency_years), 'D'))
            recency_mask = (r_dates[cand] >= recency_cut)
            # macro gate: compute cosine only on candidates
            qmac = Q_macro_n[i:i+1]                      # [1,4]
            rmac = R_macro_n[cand]                       # [M,4]
            macro_cos = (qmac @ rmac.T).ravel()
            macro_mask = (macro_cos >= tau_macro)

            mask = causal_mask & recency_mask & macro_mask
            idxs = np.where(mask)[0][:K]

            # graceful fallbacks
            if idxs.size == 0:              # relax macro first
                mask2 = causal_mask & recency_mask
                idxs = np.where(mask2)[0][:K]
            if idxs.size == 0:              # relax recency
                mask3 = causal_mask
                idxs = np.where(mask3)[0][:K]
            if idxs.size == 0:              # final: take first K
                idxs = np.arange(min(K, len(cand)))

            picked = cand[idxs]
            Z[i] = R_text_n[picked].mean(axis=0)   # TEXT pooling (mean)
            effK.append(len(picked))
            kept_frac.append(float(mask.sum()) / max(1, len(cand)))

    return Z, np.array(effK, int), np.array(kept_frac, float)


In [117]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score

num_cols = [c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"} and pd.api.types.is_numeric_dtype(train[c])]

tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
    R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
    R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

    # TRAIN z_retr (gated, causal within train)
    Ztr, effK_tr, keep_tr = z_retr_with_gates(
        q_text=R_text, q_macro=R_macro, q_dates=R_dates,
        r_text=R_text, r_macro=R_macro, r_dates=R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        batch=BATCH, use_gpu=USE_GPU
    )

    # VAL z_retr (gated, query val vs train)
    Q_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
    Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
    Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

    Zva, effK_va, keep_va = z_retr_with_gates(
        q_text=Q_text, q_macro=Q_macro, q_dates=Q_dates,
        r_text=R_text, r_macro=R_macro, r_dates=R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        batch=BATCH, use_gpu=USE_GPU
    )

    # Features: [scaled numerics+macro | text | z_retr]
    Xnum_tr = tr_df[num_cols].to_numpy(float); Xnum_va = va_df[num_cols].to_numpy(float)
    scaler = StandardScaler().fit(Xnum_tr)
    X_tr = np.hstack([scaler.transform(Xnum_tr), R_text, Ztr]).astype("float32")
    X_va = np.hstack([scaler.transform(Xnum_va), Q_text, Zva]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy(); y_va = va_df["Movement"].to_numpy()

    clf = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_tr, y_tr)
    proba_va = clf.predict_proba(X_va)[:,1]; yhat_va = (proba_va>=0.5).astype(int)

    acc = accuracy_score(y_va, yhat_va); prec = precision_score(y_va, yhat_va, zero_division=0)
    rec = recall_score(y_va, yhat_va, zero_division=0); f1 = f1_score(y_va, yhat_va, zero_division=0)
    mcc = matthews_corrcoef(y_va, yhat_va)
    try: auroc = roc_auc_score(y_va, proba_va)
    except: auroc = np.nan
    rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "val_start": va_df["Date"].min().date(), "val_end": va_df["Date"].max().date(),
        "effK_tr_med": float(np.median(effK_tr)), "keep_tr_med": float(np.median(keep_tr)),
        "effK_va_med": float(np.median(effK_va)), "keep_va_med": float(np.median(keep_va)),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
        "MCC": mcc, "AUROC": auroc, "WinRate": tm["win_rate"],
        "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"]
    })

cv_gate = pd.DataFrame(cv_rows)
print("=== Macro-Gated + Recency CV (α={}, K={}, τ_macro={}, N={}y) ===".format(ALPHA,K,TAU_MACRO,RECENCY_YEARS))
display(cv_gate)
print("\nMeans:")
display(cv_gate[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]].mean(numeric_only=True))


=== Macro-Gated + Recency CV (α=0.5, K=5, τ_macro=0.3, N=8y) ===


,fold,val_start,val_end,effK_tr_med,keep_tr_med,effK_va_med,keep_va_med,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252
0,1,2012-09-07,2015-12-03,5.0,0.240,5.0,0.256,0.641631,0.614251,0.961538,0.749625,0.297802,0.753099,0.577253,1.802142,3.347154
1,2,2015-12-04,2017-10-17,5.0,0.260,5.0,0.348,0.553648,0.548975,0.960159,0.698551,0.083705,0.613768,0.457082,0.903540,-0.547370
2,3,2017-10-18,2019-10-03,5.0,0.260,5.0,0.840,0.667382,0.648241,0.945055,0.769001,0.306498,0.782535,0.515021,1.263657,1.275976
3,4,2019-10-04,2021-08-26,5.0,0.296,5.0,0.148,0.665236,0.661654,0.926316,0.771930,0.250679,0.678123,0.564378,1.946086,3.036088
4,5,2021-08-27,2023-07-12,5.0,0.284,5.0,0.820,0.622318,0.567500,0.986957,0.720635,0.364075,0.795855,0.534335,1.478766,2.364831



Means:


Accuracy        0.630043
Precision       0.608124
Recall          0.956005
F1              0.741948
MCC             0.260552
AUROC           0.724676
WinRate         0.529614
ProfitFactor    1.478838
Sharpe_252      1.895336
dtype: float64

In [118]:
# Full-train reference blocks
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

# Train-side z_retr (causal within train) for consistent feature space
Ztr_full, _, _ = z_retr_with_gates(
    q_text=R_text_full, q_macro=R_macro_full, q_dates=R_dates_full,
    r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    batch=1024, use_gpu=USE_GPU
)

# Train final model
Xnum_full = train[[c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"}] ].to_numpy(float)
scaler_full = StandardScaler().fit(Xnum_full)
X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
y_full = train["Movement"].to_numpy()
clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

# OOD z_retr with gates (query OOD vs full train)
Z_ood, effK_ood, keep_ood = z_retr_with_gates(
    q_text=Q_text_ood, q_macro=Q_macro_ood, q_dates=Q_dates_ood,
    r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    batch=1024, use_gpu=USE_GPU
)
print("OOD effK median:", np.median(effK_ood), "| kept fraction median:", np.median(keep_ood))

Xnum_test = ood[[c for c in ood.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"}]].to_numpy(float)
X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:,1]
yhat_test = (proba_test>=0.5).astype(int)

acc = accuracy_score(y_test, yhat_test); prec = precision_score(y_test, yhat_test, zero_division=0)
rec = recall_score(y_test, yhat_test, zero_division=0); f1 = f1_score(y_test, yhat_test, zero_division=0)
mcc = matthews_corrcoef(y_test, yhat_test)
try: auroc = roc_auc_score(y_test, proba_test)
except: auroc = np.nan
rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

print("\n=== OOD — Macro-Ret + Gates (α={}, K={}, τ_macro={}, N={}y) ===".format(ALPHA,K,TAU_MACRO,RECENCY_YEARS))
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col}")


OOD effK median: 5.0 | kept fraction median: 0.948

=== OOD — Macro-Ret + Gates (α=0.5, K=5, τ_macro=0.3, N=8y) ===
Accuracy: 0.4714 | Precision: 0.4337 | Recall: 0.7347 | F1: 0.5455 | MCC: 0.0067 | AUROC: 0.4973
WinRate: 0.4405 | ProfitFactor: 0.9988 | Sharpe_252: -0.0071 | ReturnCol: Daily_Return


# Sweep τ_macro ∈ {0.2, 0.3, 0.4} and RECENCY_YEARS ∈ {8, 10}

In [120]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

ALPHA = 0.5
K = 5
BATCH = 512
USE_GPU = True
TAUS = [0.2, 0.3, 0.4]
RECENCIES = [8, 10]

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col   = "text_embed"
num_cols   = [c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"} 
              and pd.api.types.is_numeric_dtype(train[c])]

# Pre-materialize full-train / OOD arrays
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

def eval_once(tau_macro, recency_years):
    # ---------- CV (TimeSeriesSplit) ----------
    tscv = TimeSeriesSplit(n_splits=5)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
        tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

        R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
        R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
        R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

        # TRAIN z_retr (gated, causal)
        Ztr, effK_tr, keep_tr = z_retr_with_gates(
            q_text=R_text, q_macro=R_macro, q_dates=R_dates,
            r_text=R_text, r_macro=R_macro, r_dates=R_dates,
            alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
            batch=BATCH, use_gpu=USE_GPU
        )

        # VAL z_retr (gated, query val vs train)
        Q_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
        Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
        Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

        Zva, effK_va, keep_va = z_retr_with_gates(
            q_text=Q_text, q_macro=Q_macro, q_dates=Q_dates,
            r_text=R_text, r_macro=R_macro, r_dates=R_dates,
            alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
            batch=BATCH, use_gpu=USE_GPU
        )

        # Features: [scaled numerics+macro | text | z_retr]
        Xnum_tr = tr_df[num_cols].to_numpy(float); Xnum_va = va_df[num_cols].to_numpy(float)
        scaler  = StandardScaler().fit(Xnum_tr)
        X_tr = np.hstack([scaler.transform(Xnum_tr), R_text, Ztr]).astype("float32")
        X_va = np.hstack([scaler.transform(Xnum_va), Q_text, Zva]).astype("float32")
        y_tr = tr_df["Movement"].to_numpy(); y_va = va_df["Movement"].to_numpy()

        clf = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_tr, y_tr)
        proba_va = clf.predict_proba(X_va)[:,1]; yhat_va = (proba_va>=0.5).astype(int)

        acc  = accuracy_score(y_va, yhat_va)
        prec = precision_score(y_va, yhat_va, zero_division=0)
        rec  = recall_score(y_va, yhat_va, zero_division=0)
        f1   = f1_score(y_va, yhat_va, zero_division=0)
        mcc  = matthews_corrcoef(y_va, yhat_va)
        try: auroc = roc_auc_score(y_va, proba_va)
        except: auroc = np.nan
        rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
        tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

        fold_rows.append([acc,prec,rec,f1,mcc,auroc,tm["win_rate"],tm["profit_factor"],tm["sharpe_252"],
                          np.median(effK_va), np.median(keep_va)])

    cv_arr = np.array(fold_rows, dtype=float)
    cv_mean = np.nanmean(cv_arr[:, :9], axis=0)
    cv_effk_med = float(np.nanmedian(cv_arr[:, 9]))
    cv_keep_med = float(np.nanmedian(cv_arr[:,10]))

    # ---------- Full-train → OOD ----------
    # Train-side z_retr (causal within train)
    Ztr_full, _, _ = z_retr_with_gates(
        q_text=R_text_full, q_macro=R_macro_full, q_dates=R_dates_full,
        r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
        alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_full   = train[num_cols].to_numpy(float)
    scaler_full = StandardScaler().fit(Xnum_full)
    X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
    y_full = train["Movement"].to_numpy()
    clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

    # OOD z_retr with same gates
    Z_ood, effK_ood, keep_ood = z_retr_with_gates(
        q_text=Q_text_ood, q_macro=Q_macro_ood, q_dates=Q_dates_ood,
        r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
        alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_test = ood[num_cols].to_numpy(float)
    X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
    y_test = ood["Movement"].to_numpy()

    proba_test = clf_full.predict_proba(X_test)[:,1]
    yhat_test  = (proba_test>=0.5).astype(int)

    acc  = accuracy_score(y_test, yhat_test)
    prec = precision_score(y_test, yhat_test, zero_division=0)
    rec  = recall_score(y_test, yhat_test, zero_division=0)
    f1   = f1_score(y_test, yhat_test, zero_division=0)
    mcc  = matthews_corrcoef(y_test, yhat_test)
    try: auroc = roc_auc_score(y_test, proba_test)
    except: auroc = np.nan
    rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
    tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

    return {
        "tau_macro": tau_macro, "recency_y": recency_years,
        # CV
        "CV_Accuracy": cv_mean[0], "CV_Precision": cv_mean[1], "CV_Recall": cv_mean[2], "CV_F1": cv_mean[3],
        "CV_MCC": cv_mean[4], "CV_AUROC": cv_mean[5], "CV_WinRate": cv_mean[6],
        "CV_ProfitFactor": cv_mean[7], "CV_Sharpe_252": cv_mean[8],
        "CV_effK_med": cv_effk_med, "CV_keep_med": cv_keep_med,
        # OOD
        "OOD_Accuracy": acc, "OOD_Precision": prec, "OOD_Recall": rec, "OOD_F1": f1,
        "OOD_MCC": mcc, "OOD_AUROC": auroc, "OOD_WinRate": tm["win_rate"],
        "OOD_ProfitFactor": tm["profit_factor"], "OOD_Sharpe_252": tm["sharpe_252"],
        "OOD_effK_med": float(np.median(effK_ood)), "OOD_keep_med": float(np.median(keep_ood))
    }

# Run sweep
grid_results = []
for tau in TAUS:
    for nyrs in RECENCIES:
        print(f">>> Running: tau_macro={tau}, recency={nyrs}y")
        grid_results.append(eval_once(tau, nyrs))

sweep_gates = pd.DataFrame(grid_results)
print("\n=== Macro Gate + Recency sweep (α=0.5, K=5) — sorted by OOD Sharpe ===")
display(sweep_gates.sort_values("OOD_Sharpe_252", ascending=False))

print("\n(Reference columns)  tau_macro, recency_y, OOD_PF, OOD_Sharpe, OOD_AUROC, OOD_WinRate, OOD_effK_med, OOD_keep_med")
display(sweep_gates.sort_values("OOD_Sharpe_252", ascending=False)[
    ["tau_macro","recency_y","OOD_ProfitFactor","OOD_Sharpe_252","OOD_AUROC","OOD_WinRate","OOD_effK_med","OOD_keep_med"]
])


>>> Running: tau_macro=0.2, recency=8y
>>> Running: tau_macro=0.2, recency=10y
>>> Running: tau_macro=0.3, recency=8y
>>> Running: tau_macro=0.3, recency=10y
>>> Running: tau_macro=0.4, recency=8y
>>> Running: tau_macro=0.4, recency=10y

=== Macro Gate + Recency sweep (α=0.5, K=5) — sorted by OOD Sharpe ===


,tau_macro,recency_y,CV_Accuracy,CV_Precision,CV_Recall,CV_F1,CV_MCC,CV_AUROC,CV_WinRate,CV_ProfitFactor,CV_Sharpe_252,CV_effK_med,CV_keep_med,OOD_Accuracy,OOD_Precision,OOD_Recall,OOD_F1,OOD_MCC,OOD_AUROC,OOD_WinRate,OOD_ProfitFactor,OOD_Sharpe_252,OOD_effK_med,OOD_keep_med
0,0.2,8,0.630472,0.608457,0.956005,0.742174,0.261770,0.724582,0.530043,1.481725,1.901827,5.0,0.360,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497706,0.444934,1.035376,0.198787,5.0,0.952
1,0.2,10,0.630472,0.608457,0.956005,0.742174,0.261770,0.724439,0.530043,1.481725,1.901827,5.0,0.432,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497706,0.444934,1.035376,0.198787,5.0,0.992
3,0.3,10,0.629614,0.607947,0.955272,0.741581,0.259166,0.724584,0.530043,1.479124,1.896563,5.0,0.400,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497311,0.444934,1.035376,0.198787,5.0,0.984
4,0.4,8,0.630043,0.608124,0.956005,0.741948,0.260552,0.724393,0.529614,1.478838,1.895336,5.0,0.336,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497864,0.444934,1.035376,0.198787,5.0,0.932
5,0.4,10,0.629614,0.607947,0.955272,0.741581,0.259166,0.723983,0.530043,1.479124,1.896563,5.0,0.380,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497627,0.444934,1.035376,0.198787,5.0,0.964
2,0.3,8,0.630043,0.608124,0.956005,0.741948,0.260552,0.724676,0.529614,1.478838,1.895336,5.0,0.348,0.471366,0.433735,0.734694,0.545455,0.006717,0.497311,0.440529,0.998761,-0.007089,5.0,0.948



(Reference columns)  tau_macro, recency_y, OOD_PF, OOD_Sharpe, OOD_AUROC, OOD_WinRate, OOD_effK_med, OOD_keep_med


,tau_macro,recency_y,OOD_ProfitFactor,OOD_Sharpe_252,OOD_AUROC,OOD_WinRate,OOD_effK_med,OOD_keep_med
0,0.2,8,1.035376,0.198787,0.497706,0.444934,5.0,0.952
1,0.2,10,1.035376,0.198787,0.497706,0.444934,5.0,0.992
3,0.3,10,1.035376,0.198787,0.497311,0.444934,5.0,0.984
4,0.4,8,1.035376,0.198787,0.497864,0.444934,5.0,0.932
5,0.4,10,1.035376,0.198787,0.497627,0.444934,5.0,0.964
2,0.3,8,0.998761,-0.007089,0.497311,0.440529,5.0,0.948


In [121]:
# STRICTER MACRO-GATE + SHORTER RECENCY SWEEP
ALPHA = 0.5
K = 5
BATCH = 512
USE_GPU = True

TAUS_STRICT = [0.5, 0.6, 0.7]
RECENCIES_STRICT = [5, 8]

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col   = "text_embed"
num_cols   = [c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"}
              and pd.api.types.is_numeric_dtype(train[c])]

# Pre-materialize full-train / OOD arrays
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

def eval_once_strict(tau_macro, recency_years):
    # ---------- CV ----------
    tscv = TimeSeriesSplit(n_splits=5)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
        tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

        R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
        R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
        R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

        Ztr, effK_tr, keep_tr = z_retr_with_gates(
            q_text=R_text, q_macro=R_macro, q_dates=R_dates,
            r_text=R_text, r_macro=R_macro, r_dates=R_dates,
            alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
            batch=BATCH, use_gpu=USE_GPU
        )

        Q_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
        Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
        Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

        Zva, effK_va, keep_va = z_retr_with_gates(
            q_text=Q_text, q_macro=Q_macro, q_dates=Q_dates,
            r_text=R_text, r_macro=R_macro, r_dates=R_dates,
            alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
            batch=BATCH, use_gpu=USE_GPU
        )

        Xnum_tr = tr_df[num_cols].to_numpy(float); Xnum_va = va_df[num_cols].to_numpy(float)
        scaler  = StandardScaler().fit(Xnum_tr)
        X_tr = np.hstack([scaler.transform(Xnum_tr), R_text, Ztr]).astype("float32")
        X_va = np.hstack([scaler.transform(Xnum_va), Q_text, Zva]).astype("float32")
        y_tr = tr_df["Movement"].to_numpy(); y_va = va_df["Movement"].to_numpy()

        clf = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_tr, y_tr)
        proba_va = clf.predict_proba(X_va)[:,1]; yhat_va = (proba_va>=0.5).astype(int)

        acc  = accuracy_score(y_va, yhat_va)
        prec = precision_score(y_va, yhat_va, zero_division=0)
        rec  = recall_score(y_va, yhat_va, zero_division=0)
        f1   = f1_score(y_va, yhat_va, zero_division=0)
        mcc  = matthews_corrcoef(y_va, yhat_va)
        try: auroc = roc_auc_score(y_va, proba_va)
        except: auroc = np.nan
        rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
        tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

        fold_rows.append([acc,prec,rec,f1,mcc,auroc,tm["win_rate"],tm["profit_factor"],tm["sharpe_252"],
                          np.median(effK_va), np.median(keep_va)])

    cv_arr = np.array(fold_rows, dtype=float)
    cv_mean = np.nanmean(cv_arr[:, :9], axis=0)
    cv_effk_med = float(np.nanmedian(cv_arr[:, 9]))
    cv_keep_med = float(np.nanmedian(cv_arr[:,10]))

    # ---------- Full-train → OOD ----------
    Ztr_full, _, _ = z_retr_with_gates(
        q_text=R_text_full, q_macro=R_macro_full, q_dates=R_dates_full,
        r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
        alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_full   = train[num_cols].to_numpy(float)
    scaler_full = StandardScaler().fit(Xnum_full)
    X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
    y_full = train["Movement"].to_numpy()
    clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

    Z_ood, effK_ood, keep_ood = z_retr_with_gates(
        q_text=Q_text_ood, q_macro=Q_macro_ood, q_dates=Q_dates_ood,
        r_text=R_text_full, r_macro=R_macro_full, r_dates=R_dates_full,
        alpha=ALPHA, K=K, tau_macro=tau_macro, recency_years=recency_years,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_test = ood[num_cols].to_numpy(float)
    X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
    y_test = ood["Movement"].to_numpy()

    proba_test = clf_full.predict_proba(X_test)[:,1]
    yhat_test  = (proba_test>=0.5).astype(int)

    acc  = accuracy_score(y_test, yhat_test)
    prec = precision_score(y_test, yhat_test, zero_division=0)
    rec  = recall_score(y_test, yhat_test, zero_division=0)
    f1   = f1_score(y_test, yhat_test, zero_division=0)
    mcc  = matthews_corrcoef(y_test, yhat_test)
    try: auroc = roc_auc_score(y_test, proba_test)
    except: auroc = np.nan
    rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
    tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

    return {
        "tau_macro": tau_macro, "recency_y": recency_years,
        "CV_Accuracy": cv_mean[0], "CV_Precision": cv_mean[1], "CV_Recall": cv_mean[2], "CV_F1": cv_mean[3],
        "CV_MCC": cv_mean[4], "CV_AUROC": cv_mean[5], "CV_WinRate": cv_mean[6],
        "CV_ProfitFactor": cv_mean[7], "CV_Sharpe_252": cv_mean[8],
        "CV_effK_med": cv_effk_med, "CV_keep_med": cv_keep_med,
        "OOD_Accuracy": acc, "OOD_Precision": prec, "OOD_Recall": rec, "OOD_F1": f1,
        "OOD_MCC": mcc, "OOD_AUROC": auroc, "OOD_WinRate": tm["win_rate"],
        "OOD_ProfitFactor": tm["profit_factor"], "OOD_Sharpe_252": tm["sharpe_252"],
        "OOD_effK_med": float(np.median(effK_ood)), "OOD_keep_med": float(np.median(keep_ood))
    }

strict_results = []
for tau in TAUS_STRICT:
    for nyrs in RECENCIES_STRICT:
        print(f">>> Running STRICT: tau_macro={tau}, recency={nyrs}y")
        strict_results.append(eval_once_strict(tau, nyrs))

strict_df = pd.DataFrame(strict_results)
print("\n=== STRICT Macro Gate sweep (α=0.5, K=5) — sorted by OOD Sharpe ===")
display(strict_df.sort_values("OOD_Sharpe_252", ascending=False))

print("\n(Quick view) tau, recency, OOD PF, OOD Sharpe, OOD AUROC, OOD WinRate, effK_med, keep_med")
display(strict_df.sort_values("OOD_Sharpe_252", ascending=False)[
    ["tau_macro","recency_y","OOD_ProfitFactor","OOD_Sharpe_252","OOD_AUROC","OOD_WinRate","OOD_effK_med","OOD_keep_med"]
])


>>> Running STRICT: tau_macro=0.5, recency=5y
>>> Running STRICT: tau_macro=0.5, recency=8y
>>> Running STRICT: tau_macro=0.6, recency=5y
>>> Running STRICT: tau_macro=0.6, recency=8y
>>> Running STRICT: tau_macro=0.7, recency=5y
>>> Running STRICT: tau_macro=0.7, recency=8y

=== STRICT Macro Gate sweep (α=0.5, K=5) — sorted by OOD Sharpe ===


,tau_macro,recency_y,CV_Accuracy,CV_Precision,CV_Recall,CV_F1,CV_MCC,CV_AUROC,CV_WinRate,CV_ProfitFactor,CV_Sharpe_252,CV_effK_med,CV_keep_med,OOD_Accuracy,OOD_Precision,OOD_Recall,OOD_F1,OOD_MCC,OOD_AUROC,OOD_WinRate,OOD_ProfitFactor,OOD_Sharpe_252,OOD_effK_med,OOD_keep_med
1,0.5,8,0.630043,0.608124,0.956005,0.741948,0.260552,0.724455,0.529614,1.478838,1.895336,5.0,0.300,0.466960,0.431138,0.734694,0.543396,-0.001955,0.497864,0.444934,1.035376,0.198787,5.0,0.904
0,0.5,5,0.630043,0.608124,0.956005,0.741948,0.260552,0.724350,0.529614,1.478838,1.895336,5.0,0.300,0.471366,0.433735,0.734694,0.545455,0.006717,0.497785,0.440529,0.998761,-0.007089,5.0,0.212
2,0.6,5,0.630472,0.608457,0.956005,0.742174,0.261770,0.724533,0.530043,1.481725,1.901827,5.0,0.242,0.471366,0.433735,0.734694,0.545455,0.006717,0.498181,0.440529,0.998761,-0.007089,5.0,0.212
3,0.6,8,0.630472,0.608457,0.956005,0.742174,0.261770,0.724775,0.530043,1.481725,1.901827,5.0,0.242,0.471366,0.433735,0.734694,0.545455,0.006717,0.497706,0.440529,0.998761,-0.007089,5.0,0.840
4,0.7,5,0.630472,0.608457,0.956005,0.742174,0.261770,0.724217,0.530043,1.481725,1.901827,5.0,0.064,0.471366,0.433735,0.734694,0.545455,0.006717,0.498339,0.440529,0.998761,-0.007089,5.0,0.196
5,0.7,8,0.630472,0.608457,0.956005,0.742174,0.261770,0.724508,0.530043,1.481725,1.901827,5.0,0.064,0.471366,0.433735,0.734694,0.545455,0.006717,0.497390,0.440529,0.998761,-0.007089,5.0,0.728



(Quick view) tau, recency, OOD PF, OOD Sharpe, OOD AUROC, OOD WinRate, effK_med, keep_med


,tau_macro,recency_y,OOD_ProfitFactor,OOD_Sharpe_252,OOD_AUROC,OOD_WinRate,OOD_effK_med,OOD_keep_med
1,0.5,8,1.035376,0.198787,0.497864,0.444934,5.0,0.904
0,0.5,5,0.998761,-0.007089,0.497785,0.440529,5.0,0.212
2,0.6,5,0.998761,-0.007089,0.498181,0.440529,5.0,0.212
3,0.6,8,0.998761,-0.007089,0.497706,0.440529,5.0,0.840
4,0.7,5,0.998761,-0.007089,0.498339,0.440529,5.0,0.196
5,0.7,8,0.998761,-0.007089,0.497390,0.440529,5.0,0.728


# Attention Pooling instead of Mean Pooling

In [122]:
import numpy as np, faiss

# Hyperparams for attention
BETA   = 6.0   # weight on joint similarity (text ⊕ alpha*macro)
GAMMA  = 2.0   # extra weight on macro cosine
LAMBDA = 0.50  # time decay per year (higher -> prefer recent)
EPS    = 1e-9

def softmax_stable(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (np.sum(ex) + EPS)

def z_retr_with_gates_attn(
    q_text, q_macro, q_dates,
    r_text, r_macro, r_dates,
    alpha=0.5, K=5, tau_macro=0.5, recency_years=8,
    beta=BETA, gamma=GAMMA, lam=LAMBDA,
    oversample=50, batch=512, use_gpu=True
):
    """
    1) Candidate search with FAISS on joint vectors (text ; alpha*macro) using IP.
    2) Causal mask, macro gate (>= tau_macro), recency mask (>= t - N years).
    3) Attention weights = softmax( beta*joint_sim + gamma*macro_cos - lam*age_years ).
    4) z_retr = sum_i w_i * neighbor_text_i (unit-normalized text).
    Graceful fallbacks: relax macro, then recency, then take first K if still empty.
    """
    def to_unit_rows(M):
        M = M.astype("float32")
        return M / (np.linalg.norm(M, axis=1, keepdims=True) + EPS)

    def make_joint(T, M, a):
        J = np.concatenate([T.astype("float32"), (a * M.astype("float32"))], axis=1)
        return to_unit_rows(J)

    # Normalize blocks once
    R_text_n  = to_unit_rows(r_text)
    R_macro_n = to_unit_rows(r_macro)
    Q_macro_n = to_unit_rows(q_macro)
    R_joint   = make_joint(r_text, r_macro, alpha)
    Q_joint   = make_joint(q_text, q_macro, alpha)

    # Flat IP index (GPU if available)
    index = faiss.IndexFlatIP(R_joint.shape[1])
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(R_joint)

    out = np.zeros((Q_joint.shape[0], r_text.shape[1]), dtype="float32")
    effK, kept_frac, attn_entropy = [], [], []

    # constants
    DAYS_PER_Y = 365.25

    for s in range(0, Q_joint.shape[0], batch):
        e = min(s+batch, Q_joint.shape[0])
        # Oversampled candidates (get sims + ids)
        D, I = index.search(Q_joint[s:e], K*oversample)  # D: joint sims ∈ [-1,1] after norm

        for i in range(s, e):
            cand = I[i-s]            # candidate ids
            sims = D[i-s]            # joint similarities
            qd   = q_dates[i]

            # Causal & recency masks
            causal_mask = (r_dates[cand] < qd)
            recency_cut = qd - np.timedelta64(int(DAYS_PER_Y*recency_years), 'D')
            recency_mask = (r_dates[cand] >= recency_cut)

            # Macro cosine on candidates
            qmac = Q_macro_n[i:i+1]                      # [1,4]
            rmac = R_macro_n[cand]                       # [M,4]
            macro_cos = (qmac @ rmac.T).ravel()

            # Macro gate
            macro_mask = (macro_cos >= tau_macro)

            # Apply masks
            mask = causal_mask & recency_mask & macro_mask
            idxs = np.where(mask)[0]

            # Graceful relaxations
            if idxs.size == 0:
                mask2 = causal_mask & recency_mask
                idxs = np.where(mask2)[0]
            if idxs.size == 0:
                mask3 = causal_mask
                idxs = np.where(mask3)[0]
            if idxs.size == 0:
                idxs = np.arange(min(K, len(cand)))

            # Keep top-K by joint sims within idxs
            if idxs.size > K:
                # sort idxs by sims desc and take first K
                order = np.argsort(-sims[idxs])[:K]
                idxs = idxs[order]

            picked = cand[idxs]
            # Build attention weights
            # Age in years (non-negative)
            ages_years = np.maximum(0.0, (qd - r_dates[picked]).astype('timedelta64[D]').astype(np.float32) / DAYS_PER_Y)

            joint_sim = sims[idxs].astype("float32")
            macro_sel = macro_cos[idxs].astype("float32")

            score = beta*joint_sim + gamma*macro_sel - lam*ages_years
            w = softmax_stable(score)

            # Weighted text pooling
            z = (R_text_n[picked] * w[:, None]).sum(axis=0)
            # (Optional) renorm z to unit length
            nrm = np.linalg.norm(z) + EPS
            out[i] = (z / nrm).astype("float32")

            effK.append(len(picked))
            kept_frac.append(float(mask.sum()) / max(1, len(cand)))
            # Entropy for diagnostics (lower = more confident weighting)
            ent = -np.sum(w * (np.log(w + EPS)))
            attn_entropy.append(float(ent))

    return out, np.array(effK, int), np.array(kept_frac, float), np.array(attn_entropy, float)


In [123]:
TAU_MACRO = 0.5
RECENCY_YEARS = 8
ALPHA = 0.5
K = 5
BATCH = 512
USE_GPU = True

# ---------- CV ----------
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col   = "text_embed"
num_cols   = [c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"} 
              and pd.api.types.is_numeric_dtype(train[c])]

cv_rows = []
tscv = TimeSeriesSplit(n_splits=5)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    R_text  = np.vstack(tr_df[text_col].to_numpy()).astype("float32")
    R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
    R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

    Q_text  = np.vstack(va_df[text_col].to_numpy()).astype("float32")
    Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
    Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

    Ztr, effK_tr, keep_tr, Htr = z_retr_with_gates_attn(
        R_text, R_macro, R_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        batch=BATCH, use_gpu=USE_GPU
    )
    Zva, effK_va, keep_va, Hva = z_retr_with_gates_attn(
        Q_text, Q_macro, Q_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        batch=BATCH, use_gpu=USE_GPU
    )

    Xnum_tr = tr_df[num_cols].to_numpy(float); Xnum_va = va_df[num_cols].to_numpy(float)
    scaler  = StandardScaler().fit(Xnum_tr)
    X_tr = np.hstack([scaler.transform(Xnum_tr), R_text, Ztr]).astype("float32")
    X_va = np.hstack([scaler.transform(Xnum_va), Q_text, Zva]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy(); y_va = va_df["Movement"].to_numpy()

    clf = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_tr, y_tr)
    proba_va = clf.predict_proba(X_va)[:,1]; yhat_va = (proba_va>=0.5).astype(int)

    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try: auroc = roc_auc_score(y_va, proba_va)
    except: auroc = np.nan
    rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold,
        "effK_tr_med": float(np.median(effK_tr)), "keep_tr_med": float(np.median(keep_tr)), "Htr_med": float(np.median(Htr)),
        "effK_va_med": float(np.median(effK_va)), "keep_va_med": float(np.median(keep_va)), "Hva_med": float(np.median(Hva)),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"]
    })

cv_attn = pd.DataFrame(cv_rows)
print("=== CV — Macro-Ret + Gates + Attention (α=0.5, K=5, τ=0.5, N=8y) ===")
display(cv_attn)
print("\nMeans:")
display(cv_attn[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]].mean(numeric_only=True))
print("\nMedian effK/keep/entropy (lower entropy = more confident weighting):")
print("effK_va_med:", float(np.median(cv_attn["effK_va_med"])),
      "| keep_va_med:", float(np.median(cv_attn["keep_va_med"])),
      "| Hva_med:", float(np.median(cv_attn["Hva_med"])))

# ---------- OOD ----------
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

Ztr_full, _, _, _ = z_retr_with_gates_attn(
    R_text_full, R_macro_full, R_dates_full, R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    batch=1024, use_gpu=USE_GPU
)
Xnum_full   = train[num_cols].to_numpy(float)
scaler_full = StandardScaler().fit(Xnum_full)
X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
y_full = train["Movement"].to_numpy()
clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

Z_ood, effK_ood, keep_ood, H_ood = z_retr_with_gates_attn(
    Q_text_ood, Q_macro_ood, Q_dates_ood, R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    batch=1024, use_gpu=USE_GPU
)
Xnum_test = ood[num_cols].to_numpy(float)
X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:,1]
yhat_test  = (proba_test>=0.5).astype(int)

acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try: auroc = roc_auc_score(y_test, proba_test)
except: auroc = np.nan
rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

print("\n=== OOD — Macro-Ret + Gates + Attention (α=0.5, K=5, τ=0.5, N=8y) ===")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col}")
print("OOD effK_med:", float(np.median(effK_ood)), "| keep_med:", float(np.median(keep_ood)), "| attn_entropy_med:", float(np.median(H_ood)))


=== CV — Macro-Ret + Gates + Attention (α=0.5, K=5, τ=0.5, N=8y) ===


,fold,effK_tr_med,keep_tr_med,Htr_med,effK_va_med,keep_va_med,Hva_med,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252
0,1,5.0,0.196,1.586295,5.0,0.172,1.596391,0.641631,0.614251,0.961538,0.749625,0.297802,0.753305,0.577253,1.802142,3.347154
1,2,5.0,0.208,1.583442,5.0,0.300,1.605940,0.553648,0.548975,0.960159,0.698551,0.083705,0.613713,0.457082,0.903540,-0.547370
2,3,5.0,0.216,1.590074,5.0,0.780,1.596212,0.667382,0.648241,0.945055,0.769001,0.306498,0.782364,0.515021,1.263657,1.275976
3,4,5.0,0.252,1.593700,5.0,0.034,1.592315,0.667382,0.663317,0.926316,0.773060,0.256769,0.678123,0.566524,1.960520,3.068546
4,5,5.0,0.216,1.594787,5.0,0.462,1.602724,0.622318,0.567500,0.986957,0.720635,0.364075,0.794713,0.534335,1.478766,2.364831



Means:


Accuracy        0.630472
Precision       0.608457
Recall          0.956005
F1              0.742174
MCC             0.261770
AUROC           0.724444
WinRate         0.530043
ProfitFactor    1.481725
Sharpe_252      1.901827
dtype: float64


Median effK/keep/entropy (lower entropy = more confident weighting):
effK_va_med: 5.0 | keep_va_med: 0.3 | Hva_med: 1.596390724182129

=== OOD — Macro-Ret + Gates + Attention (α=0.5, K=5, τ=0.5, N=8y) ===
Accuracy: 0.4670 | Precision: 0.4311 | Recall: 0.7347 | F1: 0.5434 | MCC: -0.0020 | AUROC: 0.4983
WinRate: 0.4449 | ProfitFactor: 1.0354 | Sharpe_252: 0.1988 | ReturnCol: Daily_Return
OOD effK_med: 5.0 | keep_med: 0.904 | attn_entropy_med: 1.5583785772323608


In [124]:
# Sweep attention weights: beta (joint sim), gamma (macro), lambda (time decay)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score

# Fixed gates (best so far)
ALPHA, K = 0.5, 5
TAU_MACRO, RECENCY_YEARS = 0.5, 8
BATCH, USE_GPU = 512, True

BETAS  = [4.0, 6.0, 8.0]
GAMMAS = [1.0, 2.0, 3.0]
LAMBDAS= [0.3, 0.5, 0.8]

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
text_col   = "text_embed"
num_cols   = [c for c in train.columns if c not in {"Date","Movement","Daily_Return","text_embed","z_retr"}
              and pd.api.types.is_numeric_dtype(train[c])]

# Pre-materialize arrays
R_text_full  = np.vstack(train[text_col].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

Q_text_ood  = np.vstack(ood[text_col].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

# Fit scaler and full-train classifier once per setting (since Ztr depends on attn weights)
def eval_settings(beta, gamma, lam):
    # Train-side z_retr (attn + gates) for feature space
    Ztr_full, _, _, _ = z_retr_with_gates_attn(
        R_text_full, R_macro_full, R_dates_full,
        R_text_full, R_macro_full, R_dates_full,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=beta, gamma=gamma, lam=lam,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_full   = train[num_cols].to_numpy(float)
    scaler_full = StandardScaler().fit(Xnum_full)
    X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
    y_full = train["Movement"].to_numpy()
    clf_full = LogisticRegression(max_iter=3000, solver="liblinear", n_jobs=1, random_state=42).fit(X_full, y_full)

    # OOD z_retr with same attn weights
    Z_ood, effK_ood, keep_ood, H_ood = z_retr_with_gates_attn(
        Q_text_ood, Q_macro_ood, Q_dates_ood,
        R_text_full, R_macro_full, R_dates_full,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=beta, gamma=gamma, lam=lam,
        batch=1024, use_gpu=USE_GPU
    )
    Xnum_test = ood[num_cols].to_numpy(float)
    X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
    y_test = ood["Movement"].to_numpy()

    proba_test = clf_full.predict_proba(X_test)[:,1]
    yhat_test  = (proba_test>=0.5).astype(int)

    acc  = accuracy_score(y_test, yhat_test)
    prec = precision_score(y_test, yhat_test, zero_division=0)
    rec  = recall_score(y_test, yhat_test, zero_division=0)
    f1   = f1_score(y_test, yhat_test, zero_division=0)
    mcc  = matthews_corrcoef(y_test, yhat_test)
    try: auroc = roc_auc_score(y_test, proba_test)
    except: auroc = np.nan

    rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
    tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

    return {
        "beta": beta, "gamma": gamma, "lambda": lam,
        "OOD_Accuracy": acc, "OOD_Precision": prec, "OOD_Recall": rec, "OOD_F1": f1,
        "OOD_MCC": mcc, "OOD_AUROC": auroc, "OOD_WinRate": tm["win_rate"],
        "OOD_ProfitFactor": tm["profit_factor"], "OOD_Sharpe_252": tm["sharpe_252"],
        "OOD_effK_med": float(np.median(effK_ood)), "OOD_keep_med": float(np.median(keep_ood)),
        "OOD_attn_entropy_med": float(np.median(H_ood))
    }

results = []
for b in BETAS:
    for g in GAMMAS:
        for l in LAMBDAS:
            print(f">>> OOD sweep: beta={b}, gamma={g}, lambda={l}")
            results.append(eval_settings(b, g, l))

attn_sweep = pd.DataFrame(results)
print("\n=== Attention weight sweep (fixed τ=0.5, N=8y, α=0.5, K=5) — sorted by OOD Sharpe ===")
display(attn_sweep.sort_values("OOD_Sharpe_252", ascending=False)[
    ["beta","gamma","lambda","OOD_ProfitFactor","OOD_Sharpe_252","OOD_AUROC","OOD_WinRate","OOD_effK_med","OOD_keep_med","OOD_attn_entropy_med"]
])


>>> OOD sweep: beta=4.0, gamma=1.0, lambda=0.3
>>> OOD sweep: beta=4.0, gamma=1.0, lambda=0.5
>>> OOD sweep: beta=4.0, gamma=1.0, lambda=0.8
>>> OOD sweep: beta=4.0, gamma=2.0, lambda=0.3
>>> OOD sweep: beta=4.0, gamma=2.0, lambda=0.5
>>> OOD sweep: beta=4.0, gamma=2.0, lambda=0.8
>>> OOD sweep: beta=4.0, gamma=3.0, lambda=0.3
>>> OOD sweep: beta=4.0, gamma=3.0, lambda=0.5
>>> OOD sweep: beta=4.0, gamma=3.0, lambda=0.8
>>> OOD sweep: beta=6.0, gamma=1.0, lambda=0.3
>>> OOD sweep: beta=6.0, gamma=1.0, lambda=0.5
>>> OOD sweep: beta=6.0, gamma=1.0, lambda=0.8
>>> OOD sweep: beta=6.0, gamma=2.0, lambda=0.3
>>> OOD sweep: beta=6.0, gamma=2.0, lambda=0.5
>>> OOD sweep: beta=6.0, gamma=2.0, lambda=0.8
>>> OOD sweep: beta=6.0, gamma=3.0, lambda=0.3
>>> OOD sweep: beta=6.0, gamma=3.0, lambda=0.5
>>> OOD sweep: beta=6.0, gamma=3.0, lambda=0.8
>>> OOD sweep: beta=8.0, gamma=1.0, lambda=0.3
>>> OOD sweep: beta=8.0, gamma=1.0, lambda=0.5
>>> OOD sweep: beta=8.0, gamma=1.0, lambda=0.8
>>> OOD sweep

,beta,gamma,lambda,OOD_ProfitFactor,OOD_Sharpe_252,OOD_AUROC,OOD_WinRate,OOD_effK_med,OOD_keep_med,OOD_attn_entropy_med
0,4.0,1.0,0.3,1.035376,0.198787,0.497943,0.444934,5.0,0.904,1.589785
14,6.0,2.0,0.8,1.035376,0.198787,0.498497,0.444934,5.0,0.904,1.502208
25,8.0,3.0,0.5,1.035376,0.198787,0.498181,0.444934,5.0,0.904,1.549660
24,8.0,3.0,0.3,1.035376,0.198787,0.498022,0.444934,5.0,0.904,1.583765
23,8.0,2.0,0.8,1.035376,0.198787,0.498339,0.444934,5.0,0.904,1.502190
22,8.0,2.0,0.5,1.035376,0.198787,0.498260,0.444934,5.0,0.904,1.558384
21,8.0,2.0,0.3,1.035376,0.198787,0.498102,0.444934,5.0,0.904,1.586645
20,8.0,1.0,0.8,1.035376,0.198787,0.498339,0.444934,5.0,0.904,1.509882
19,8.0,1.0,0.5,1.035376,0.198787,0.498181,0.444934,5.0,0.904,1.566477
18,8.0,1.0,0.3,1.035376,0.198787,0.497627,0.444934,5.0,0.904,1.589797


# Adding small MLP

In [128]:
# === Retrieval gates (best so far) ===
ALPHA = 0.5        # text ⊕ alpha*macro
K = 5
TAU_MACRO = 0.5
RECENCY_YEARS = 8

# === Attention weights (keep what worked) ===
BETA, GAMMA, LAMBDA = 6.0, 2.0, 0.5

# === Time-safe MLP hyperparams ===
from sklearn.neural_network import MLPClassifier
MLP_KW = dict(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,              # stronger L2 for stability
    learning_rate_init=5e-4,
    batch_size=128,
    max_iter=1000,
    early_stopping=False,    # ✔ avoid random val leakage
    shuffle=False,           # ✔ keep order for TS
    n_iter_no_change=20,
    random_state=42,
    verbose=False
)

BATCH = 512
USE_GPU = True

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
TEXT_COL  = "text_embed"
# keep numerics only (no Date/Movement/returns/text/z_retr)
NUM_COLS  = [c for c in train.columns
             if c not in {"Date","Movement","Daily_Return",TEXT_COL,"z_retr"}
             and pd.api.types.is_numeric_dtype(train[c])]


In [129]:
import numpy as np

def pick_threshold_by_metric(y_true, proba, returns=None, metric="pf"):
    """
    Sweep thresholds, choose the one that maximizes the chosen metric.
    metric: "pf" | "sharpe" | "f1"
    """
    grid = np.linspace(0.3, 0.7, 21)  # widen/narrow if needed
    best_t, best_val = 0.5, -1e9
    for t in grid:
        yhat = (proba >= t).astype(int)
        if metric in ("pf","sharpe"):
            m = trading_metrics(y_true, yhat, returns)[
                "profit_factor" if metric=="pf" else "sharpe_252"
            ]
        else:
            from sklearn.metrics import f1_score
            m = f1_score(y_true, yhat, zero_division=0)
        if np.isfinite(m) and m > best_val:
            best_val, best_t = m, t
    return best_t, best_val


In [130]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

cv_rows, best_ts = [], []
tscv = TimeSeriesSplit(n_splits=5)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    # Reference (train fold)
    R_text  = np.vstack(tr_df[TEXT_COL].to_numpy()).astype("float32")
    R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
    R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")

    # Queries (val fold)
    Q_text  = np.vstack(va_df[TEXT_COL].to_numpy()).astype("float32")
    Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
    Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

    # Attention retrieval (gates + recency)
    Ztr, effK_tr, keep_tr, Htr = z_retr_with_gates_attn(
        R_text, R_macro, R_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=BETA, gamma=GAMMA, lam=LAMBDA,
        batch=BATCH, use_gpu=USE_GPU
    )
    Zva, effK_va, keep_va, Hva = z_retr_with_gates_attn(
        Q_text, Q_macro, Q_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=BETA, gamma=GAMMA, lam=LAMBDA,
        batch=BATCH, use_gpu=USE_GPU
    )

    # Features: scale numerics; concat [scaled numerics | text | z_retr]
    Xnum_tr = tr_df[NUM_COLS].to_numpy(float)
    Xnum_va = va_df[NUM_COLS].to_numpy(float)
    scaler  = StandardScaler().fit(Xnum_tr)

    X_tr = np.hstack([scaler.transform(Xnum_tr), R_text, Ztr]).astype("float32")
    X_va = np.hstack([scaler.transform(Xnum_va), Q_text, Zva]).astype("float32")
    y_tr = tr_df["Movement"].to_numpy()
    y_va = va_df["Movement"].to_numpy()

    # Train MLP (time-safe)
    clf = MLPClassifier(**MLP_KW)
    clf.fit(X_tr, y_tr)

    proba_va = clf.predict_proba(X_va)[:,1]

    # Choose threshold on *validation fold* using trading metric
    rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
    best_t, _ = pick_threshold_by_metric(
        y_va, proba_va,
        va_df[rr_col].to_numpy() if rr_col else None,
        metric="pf"   # or "sharpe" if you prefer
    )
    best_ts.append(best_t)
    yhat_va  = (proba_va >= best_t).astype(int)

    # Metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try: auroc = roc_auc_score(y_va, proba_va)
    except: auroc = np.nan
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold,
        "thr": best_t,
        "effK_tr_med": float(np.median(effK_tr)),
        "keep_tr_med": float(np.median(keep_tr)),
        "Htr_med": float(np.median(Htr)),
        "effK_va_med": float(np.median(effK_va)),
        "keep_va_med": float(np.median(keep_va)),
        "Hva_med": float(np.median(Hva)),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
        "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"]
    })

cv_mlp = pd.DataFrame(cv_rows)
thr_cv = float(np.median(best_ts))

print("=== CV — Macro-Ret + Gates + Attention + MLP (time-safe, tuned threshold) ===")
display(cv_mlp)
print("\n=== CV means ===")
display(cv_mlp[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]].mean(numeric_only=True))
print(f"\nChosen probability threshold from CV (median): {thr_cv:.3f}")
print("\nDiagnostics (median): effK_va, keep_va, entropy_va")
print(float(np.median(cv_mlp["effK_va_med"])),
      float(np.median(cv_mlp["keep_va_med"])),
      float(np.median(cv_mlp["Hva_med"])))


=== CV — Macro-Ret + Gates + Attention + MLP (time-safe, tuned threshold) ===


,fold,thr,effK_tr_med,keep_tr_med,Htr_med,effK_va_med,keep_va_med,Hva_med,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252
0,1,0.34,5.0,0.196,1.586295,5.0,0.172,1.596391,0.487124,0.594595,0.253846,0.355795,0.041271,0.521042,0.560086,1.711983,3.058898
1,2,0.70,5.0,0.208,1.583442,5.0,0.300,1.605940,0.495708,0.535398,0.482072,0.507338,-0.006284,0.510868,0.532189,1.129526,0.657179
2,3,0.52,5.0,0.216,1.590074,5.0,0.780,1.596212,0.506438,0.571906,0.626374,0.597902,-0.037842,0.452922,0.478541,0.854096,-0.860570
3,4,0.68,5.0,0.252,1.593700,5.0,0.034,1.592315,0.577253,0.618280,0.807018,0.700152,0.027312,0.517650,0.532189,1.109045,0.480318
4,5,0.60,5.0,0.216,1.594787,5.0,0.462,1.602724,0.497854,0.494709,0.813043,0.615132,0.004754,0.506706,0.474249,0.943079,-0.354837



=== CV means ===


Accuracy        0.512876
Precision       0.562978
Recall          0.596471
F1              0.555264
MCC             0.005842
AUROC           0.501838
WinRate         0.515451
ProfitFactor    1.149545
Sharpe_252      0.596198
dtype: float64


Chosen probability threshold from CV (median): 0.600

Diagnostics (median): effK_va, keep_va, entropy_va
5.0 0.3 1.596390724182129


In [131]:
# Full train references
R_text_full  = np.vstack(train[TEXT_COL].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")

# OOD queries
Q_text_ood  = np.vstack(ood[TEXT_COL].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

# Train-side z_retr for feature space
Ztr_full, _, _, _ = z_retr_with_gates_attn(
    R_text_full, R_macro_full, R_dates_full,
    R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    beta=BETA, gamma=GAMMA, lam=LAMBDA,
    batch=1024, use_gpu=USE_GPU
)

# Train final MLP on full train
from sklearn.preprocessing import StandardScaler
Xnum_full   = train[NUM_COLS].to_numpy(float)
scaler_full = StandardScaler().fit(Xnum_full)
X_full = np.hstack([scaler_full.transform(Xnum_full), R_text_full, Ztr_full]).astype("float32")
y_full = train["Movement"].to_numpy()

clf_full = MLPClassifier(**MLP_KW)
clf_full.fit(X_full, y_full)

# OOD z_retr (same gates/attention)
Z_ood, effK_ood, keep_ood, H_ood = z_retr_with_gates_attn(
    Q_text_ood, Q_macro_ood, Q_dates_ood,
    R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    beta=BETA, gamma=GAMMA, lam=LAMBDA,
    batch=1024, use_gpu=USE_GPU
)
Xnum_test = ood[NUM_COLS].to_numpy(float)
X_test = np.hstack([scaler_full.transform(Xnum_test), Q_text_ood, Z_ood]).astype("float32")
y_test = ood["Movement"].to_numpy()

proba_test = clf_full.predict_proba(X_test)[:,1]
yhat_test  = (proba_test >= thr_cv).astype(int)  # ✔ apply CV-tuned threshold

# OOD metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try: auroc = roc_auc_score(y_test, proba_test)
except: auroc = np.nan

rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

print("\n=== OOD — Macro-Ret + Gates + Attention + MLP (time-safe, CV-tuned thr) ===")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col}")
print("OOD effK_med:", float(np.median(effK_ood)),
      "| keep_med:", float(np.median(keep_ood)),
      "| attn_entropy_med:", float(np.median(H_ood)))



=== OOD — Macro-Ret + Gates + Attention + MLP (time-safe, CV-tuned thr) ===
Accuracy: 0.5683 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000 | MCC: 0.0000 | AUROC: 0.4685
WinRate: 0.5991 | ProfitFactor: 1.4956 | Sharpe_252: 2.2949 | ReturnCol: Daily_Return
OOD effK_med: 5.0 | keep_med: 0.904 | attn_entropy_med: 1.5583785772323608


# KNN Prior Feature + LightGBM

In [133]:
import numpy as np, faiss

EPS = 1e-9
DAYS_PER_Y = 365.25

def _to_unit_rows(M):
    M = M.astype("float32")
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + EPS)

def _make_joint(T, M, alpha):
    J = np.concatenate([T.astype("float32"), (alpha * M.astype("float32"))], axis=1)
    return _to_unit_rows(J)

def _softmax_stable(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (np.sum(ex) + EPS)

def z_retr_with_neighbors_attn(
    q_text, q_macro, q_dates,
    r_text, r_macro, r_dates,
    alpha=0.5, K=5, tau_macro=0.5, recency_years=8,
    beta=6.0, gamma=2.0, lam=0.5,
    oversample=50, batch=512, use_gpu=True
):
    """
    Returns:
      z_retr: (T, d_text) attention-weighted pooled neighbor text vectors
      effK, kept_frac, attn_entropy: diagnostics
      neigh_ids: list of arrays, each shape (k_i,) with selected neighbor row indices
      neigh_wts: list of arrays, attention weights aligned to neigh_ids
    """
    R_text_n  = _to_unit_rows(r_text)
    R_macro_n = _to_unit_rows(r_macro)
    Q_macro_n = _to_unit_rows(q_macro)

    R_joint = _make_joint(r_text, r_macro, alpha)
    Q_joint = _make_joint(q_text, q_macro, alpha)

    index = faiss.IndexFlatIP(R_joint.shape[1])
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(R_joint)

    T, d = Q_joint.shape[0], r_text.shape[1]
    z_out = np.zeros((T, d), dtype="float32")
    effK, kept_frac, attn_entropy = [], [], []
    neigh_ids, neigh_wts = [], []

    for s in range(0, T, batch):
        e = min(s+batch, T)
        D, I = index.search(Q_joint[s:e], K*oversample)  # candidates

        for i in range(s, e):
            cand = I[i-s]
            sims = D[i-s]
            qd   = q_dates[i]

            # masks
            causal_mask  = (r_dates[cand] < qd)
            recency_cut  = qd - np.timedelta64(int(DAYS_PER_Y*recency_years), 'D')
            recency_mask = (r_dates[cand] >= recency_cut)

            qmac = Q_macro_n[i:i+1]
            rmac = R_macro_n[cand]
            macro_cos  = (qmac @ rmac.T).ravel()
            macro_mask = (macro_cos >= tau_macro)

            mask = causal_mask & recency_mask & macro_mask
            idxs = np.where(mask)[0]
            if idxs.size == 0:
                idxs = np.where(causal_mask & recency_mask)[0]
            if idxs.size == 0:
                idxs = np.where(causal_mask)[0]
            if idxs.size == 0:
                idxs = np.arange(min(K, len(cand)))

            # top-K by joint sim
            if idxs.size > K:
                order = np.argsort(-sims[idxs])[:K]
                idxs = idxs[order]

            picked = cand[idxs]
            ages_years = np.maximum(
                0.0, (qd - r_dates[picked]).astype('timedelta64[D]').astype(np.float32) / DAYS_PER_Y
            )
            joint_sim = sims[idxs].astype("float32")
            macro_sel = macro_cos[idxs].astype("float32")

            score = beta*joint_sim + gamma*macro_sel - lam*ages_years
            w = _softmax_stable(score)  # attention weights

            z  = (R_text_n[picked] * w[:, None]).sum(axis=0)
            z /= (np.linalg.norm(z) + EPS)

            z_out[i] = z.astype("float32")

            effK.append(len(picked))
            kept_frac.append(float(mask.sum()) / max(1, len(cand)))
            attn_entropy.append(float(-np.sum(w * (np.log(w + EPS)))))
            neigh_ids.append(picked)
            neigh_wts.append(w)

    return z_out, np.array(effK), np.array(kept_frac), np.array(attn_entropy), neigh_ids, neigh_wts

def build_knn_priors(neigh_ids, neigh_wts, ref_outcomes, ref_returns=None):
    """
    ref_outcomes: (N_ref,) array of 0/1 Movement for reference (train) rows
    ref_returns : (N_ref,) array of Daily_Return for reference (optional)
    Returns:
      p_knn: attention-weighted mean UP probability (0..1)
      r_knn: attention-weighted mean return (can be None if ref_returns is None)
    """
    T = len(neigh_ids)
    p = np.zeros(T, dtype="float32")
    r = np.zeros(T, dtype="float32") if ref_returns is not None else None

    for i in range(T):
        ids = neigh_ids[i]; w = neigh_wts[i]
        p[i] = float((ref_outcomes[ids] * w).sum())
        if r is not None:
            r[i] = float((ref_returns[ids] * w).sum())

    return p, r


In [135]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

# Config
ALPHA, K = 0.5, 5
TAU_MACRO, RECENCY_YEARS = 0.5, 8
BETA, GAMMA, LAMBDA = 6.0, 2.0, 0.5
BATCH, USE_GPU = 1024, True

MACRO_COLS = ["cpi_yoy_lagged_z","unrate_lagged_z","t10y2y_lagged_z","gdp_qoq_lagged_z"]
TEXT_COL   = "text_embed"
NUM_COLS   = [c for c in train.columns
              if c not in {"Date","Movement","Daily_Return",TEXT_COL,"z_retr"}
              and pd.api.types.is_numeric_dtype(train[c])]

LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    n_estimators=800,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1e-3,
    reg_lambda=1e-2,
    random_state=42
)

def pick_threshold_by_metric(y_true, proba, returns=None, metric="pf"):
    grid = np.linspace(0.3, 0.7, 21)
    best_t, best_val = 0.5, -1e9
    for t in grid:
        yhat = (proba >= t).astype(int)
        if metric in ("pf","sharpe"):
            m = trading_metrics(y_true, yhat, returns)[
                "profit_factor" if metric=="pf" else "sharpe_252"
            ]
        else:
            from sklearn.metrics import f1_score
            m = f1_score(y_true, yhat, zero_division=0)
        if np.isfinite(m) and m > best_val:
            best_val, best_t = m, t
    return best_t, best_val

# --- CV ---
tscv = TimeSeriesSplit(n_splits=5)
cv_rows, best_ts = [], []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()

    # Reference (train fold)
    R_text  = np.vstack(tr_df[TEXT_COL].to_numpy()).astype("float32")
    R_macro = tr_df[MACRO_COLS].to_numpy().astype("float32")
    R_dates = tr_df["Date"].to_numpy(dtype="datetime64[ns]")
    R_y     = tr_df["Movement"].to_numpy().astype("float32")
    R_ret   = tr_df["Daily_Return"].to_numpy().astype("float32") if "Daily_Return" in tr_df else None

    # Queries (validation fold)
    Q_text  = np.vstack(va_df[TEXT_COL].to_numpy()).astype("float32")
    Q_macro = va_df[MACRO_COLS].to_numpy().astype("float32")
    Q_dates = va_df["Date"].to_numpy(dtype="datetime64[ns]")

    # Retrieval with neighbors
    Ztr, effK_tr, keep_tr, Htr, ids_tr, w_tr = z_retr_with_neighbors_attn(
        R_text, R_macro, R_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=BETA, gamma=GAMMA, lam=LAMBDA, batch=BATCH, use_gpu=USE_GPU
    )
    Zva, effK_va, keep_va, Hva, ids_va, w_va = z_retr_with_neighbors_attn(
        Q_text, Q_macro, Q_dates, R_text, R_macro, R_dates,
        alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
        beta=BETA, gamma=GAMMA, lam=LAMBDA, batch=BATCH, use_gpu=USE_GPU
    )

    # kNN priors from train outcomes/returns
    p_knn_tr, r_knn_tr = build_knn_priors(ids_tr, w_tr, ref_outcomes=R_y, ref_returns=R_ret)
    p_knn_va, r_knn_va = build_knn_priors(ids_va, w_va, ref_outcomes=R_y, ref_returns=R_ret)

    # Features
    Xnum_tr = tr_df[NUM_COLS].to_numpy(float)
    Xnum_va = va_df[NUM_COLS].to_numpy(float)
    scaler  = StandardScaler().fit(Xnum_tr)

    # concat: [scaled numerics | text | z_retr | p_knn | r_knn]
    X_tr = np.hstack([
        scaler.transform(Xnum_tr),
        R_text, Ztr,
        p_knn_tr[:,None],
        (r_knn_tr[:,None] if r_knn_tr is not None else np.zeros((len(p_knn_tr),1)))
    ]).astype("float32")

    X_va = np.hstack([
        scaler.transform(Xnum_va),
        Q_text, Zva,
        p_knn_va[:,None],
        (r_knn_va[:,None] if r_knn_va is not None else np.zeros((len(p_knn_va),1)))
    ]).astype("float32")

    y_tr = tr_df["Movement"].to_numpy()
    y_va = va_df["Movement"].to_numpy()

    # LightGBM (time-safe: train on tr_df, validate on future va_df)
    clf = lgb.LGBMClassifier(**LGB_PARAMS)
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc")

    proba_va = clf.predict_proba(X_va)[:,1]

    # Tune threshold on validation fold by PF (or "sharpe")
    rr_col = "Daily_Return" if "Daily_Return" in va_df.columns else None
    best_t, _ = pick_threshold_by_metric(y_va, proba_va,
                                         va_df[rr_col].to_numpy() if rr_col else None,
                                         metric="pf")
    best_ts.append(best_t)
    yhat_va  = (proba_va >= best_t).astype(int)

    # Metrics
    acc  = accuracy_score(y_va, yhat_va)
    prec = precision_score(y_va, yhat_va, zero_division=0)
    rec  = recall_score(y_va, yhat_va, zero_division=0)
    f1   = f1_score(y_va, yhat_va, zero_division=0)
    mcc  = matthews_corrcoef(y_va, yhat_va)
    try: auroc = roc_auc_score(y_va, proba_va)
    except: auroc = np.nan
    tm = trading_metrics(y_va, yhat_va, va_df[rr_col].to_numpy() if rr_col else None)

    cv_rows.append({
        "fold": fold, "thr": best_t,
        "effK_tr_med": float(np.median(effK_tr)), "keep_tr_med": float(np.median(keep_tr)), "Htr_med": float(np.median(Htr)),
        "effK_va_med": float(np.median(effK_va)), "keep_va_med": float(np.median(keep_va)), "Hva_med": float(np.median(Hva)),
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc, "AUROC": auroc,
        "WinRate": tm["win_rate"], "ProfitFactor": tm["profit_factor"], "Sharpe_252": tm["sharpe_252"]
    })

cv_lgb = pd.DataFrame(cv_rows)
thr_cv = float(np.median(best_ts))

print("=== CV — Macro-Ret + Attention + kNN Priors + LightGBM ===")
display(cv_lgb)
print("\n=== CV means ===")
display(cv_lgb[["Accuracy","Precision","Recall","F1","MCC","AUROC","WinRate","ProfitFactor","Sharpe_252"]].mean(numeric_only=True))
print(f"\nChosen probability threshold from CV (median): {thr_cv:.3f}")
print("\nDiagnostics (median): effK_va, keep_va, entropy_va")
print(float(np.median(cv_lgb["effK_va_med"])),
      float(np.median(cv_lgb["keep_va_med"])),
      float(np.median(cv_lgb["Hva_med"])))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[LightGBM] [Info] Number of positive: 231, number of negative: 236
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002879 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 121675
[LightGBM] [Info] Number of data points in the train set: 467, number of used features: 782
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.494647 -> initscore=-0.021414
[LightGBM] [Info] Start training from score -0.021414
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,fold,thr,effK_tr_med,keep_tr_med,Htr_med,effK_va_med,keep_va_med,Hva_med,Accuracy,Precision,Recall,F1,MCC,AUROC,WinRate,ProfitFactor,Sharpe_252
0,1,0.54,5.0,0.196,1.586295,5.0,0.172,1.596391,0.493562,0.625000,0.230769,0.337079,0.068780,0.508607,0.519313,1.280026,1.410651
1,2,0.68,5.0,0.208,1.583442,5.0,0.300,1.605940,0.504292,0.536232,0.589641,0.561670,-0.005790,0.499639,0.519313,1.207248,1.015701
2,3,0.36,5.0,0.216,1.590074,5.0,0.780,1.596212,0.527897,0.583072,0.681319,0.628378,-0.008269,0.487123,0.508584,1.153094,0.777370
3,4,0.38,5.0,0.252,1.593700,5.0,0.034,1.592315,0.564378,0.618497,0.750877,0.678288,0.024071,0.510071,0.532189,1.261650,1.076726
4,5,0.70,5.0,0.216,1.594787,5.0,0.462,1.602724,0.517167,0.508197,0.673913,0.579439,0.040288,0.514480,0.493562,0.976373,-0.144778



=== CV means ===


Accuracy        0.521459
Precision       0.574200
Recall          0.585304
F1              0.556971
MCC             0.023816
AUROC           0.503984
WinRate         0.514592
ProfitFactor    1.175678
Sharpe_252      0.827134
dtype: float64


Chosen probability threshold from CV (median): 0.540

Diagnostics (median): effK_va, keep_va, entropy_va
5.0 0.3 1.596390724182129


In [137]:
# Full reference (train)
R_text_full  = np.vstack(train[TEXT_COL].to_numpy()).astype("float32")
R_macro_full = train[MACRO_COLS].to_numpy().astype("float32")
R_dates_full = train["Date"].to_numpy(dtype="datetime64[ns]")
R_y_full     = train["Movement"].to_numpy().astype("float32")
R_ret_full   = train["Daily_Return"].to_numpy().astype("float32") if "Daily_Return" in train else None

# OOD queries
Q_text_ood  = np.vstack(ood[TEXT_COL].to_numpy()).astype("float32")
Q_macro_ood = ood[MACRO_COLS].to_numpy().astype("float32")
Q_dates_ood = ood["Date"].to_numpy(dtype="datetime64[ns]")

# Train-side retrieval (to compute z_retr & priors for training rows)
Ztr_full, _, _, _, ids_tr_full, w_tr_full = z_retr_with_neighbors_attn(
    R_text_full, R_macro_full, R_dates_full,
    R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    beta=BETA, gamma=GAMMA, lam=LAMBDA, batch=BATCH, use_gpu=USE_GPU
)
p_knn_tr_full, r_knn_tr_full = build_knn_priors(ids_tr_full, w_tr_full, R_y_full, R_ret_full)

# OOD retrieval (neighbors from the train set)
Z_ood, effK_ood, keep_ood, H_ood, ids_ood, w_ood = z_retr_with_neighbors_attn(
    Q_text_ood, Q_macro_ood, Q_dates_ood,
    R_text_full, R_macro_full, R_dates_full,
    alpha=ALPHA, K=K, tau_macro=TAU_MACRO, recency_years=RECENCY_YEARS,
    beta=BETA, gamma=GAMMA, lam=LAMBDA, batch=BATCH, use_gpu=USE_GPU
)
p_knn_ood, r_knn_ood = build_knn_priors(ids_ood, w_ood, R_y_full, R_ret_full)

# Build features
from sklearn.preprocessing import StandardScaler
Xnum_full   = train[NUM_COLS].to_numpy(float)
scaler_full = StandardScaler().fit(Xnum_full)

X_full = np.hstack([
    scaler_full.transform(Xnum_full),
    R_text_full, Ztr_full,
    p_knn_tr_full[:,None],
    (r_knn_tr_full[:,None] if r_knn_tr_full is not None else np.zeros((len(p_knn_tr_full),1)))
]).astype("float32")
y_full = train["Movement"].to_numpy()

Xnum_test = ood[NUM_COLS].to_numpy(float)
X_test = np.hstack([
    scaler_full.transform(Xnum_test),
    Q_text_ood, Z_ood,
    p_knn_ood[:,None],
    (r_knn_ood[:,None] if r_knn_ood is not None else np.zeros((len(p_knn_ood),1)))
]).astype("float32")
y_test = ood["Movement"].to_numpy()

# Train LightGBM on full train
clf_full = lgb.LGBMClassifier(**LGB_PARAMS)
clf_full.fit(X_full, y_full)

# Predict OOD with CV-tuned threshold
proba_test = clf_full.predict_proba(X_test)[:,1]
yhat_test  = (proba_test >= thr_cv).astype(int)

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
acc  = accuracy_score(y_test, yhat_test)
prec = precision_score(y_test, yhat_test, zero_division=0)
rec  = recall_score(y_test, yhat_test, zero_division=0)
f1   = f1_score(y_test, yhat_test, zero_division=0)
mcc  = matthews_corrcoef(y_test, yhat_test)
try: auroc = roc_auc_score(y_test, proba_test)
except: auroc = np.nan

rr_col = "Daily_Return" if "Daily_Return" in ood.columns else None
tm = trading_metrics(y_test, yhat_test, ood[rr_col].to_numpy() if rr_col else None)

print("\n=== OOD — Macro-Ret + Attention + kNN Priors + LightGBM (CV-tuned thr) ===")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}")
if rr_col:
    print(f"WinRate: {tm['win_rate']:.4f} | ProfitFactor: {tm['profit_factor']:.4f} | Sharpe_252: {tm['sharpe_252']:.4f} | ReturnCol: {rr_col}")
print("OOD effK_med:", float(np.median(effK_ood)),
      "| keep_med:", float(np.median(keep_ood)),
      "| attn_entropy_med:", float(np.median(H_ood)))


[LightGBM] [Info] Number of positive: 1530, number of negative: 1267
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005682 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 198886
[LightGBM] [Info] Number of data points in the train set: 2797, number of used features: 782
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.547015 -> initscore=0.188616
[LightGBM] [Info] Start training from score 0.188616

=== OOD — Macro-Ret + Attention + kNN Priors + LightGBM (CV-tuned thr) ===
Accuracy: 0.4317 | Precision: 0.4317 | Recall: 1.0000 | F1: 0.6031 | MCC: 0.0000 | AUROC: 0.5254
WinRate: 0.4009 | ProfitFactor: 0.6686 | Sharpe_252: -2.2949 | ReturnCol: Daily_Return
OOD effK_med: 5.0 | keep_med: 0.904 | attn_entropy_med: 1.5583785772323608
